# 06 · F3, distillation, first ensembles

Oracles for the hardest family F3, detectors, β-loss, distillation, counters, decoding experiments and the first 3-model ensemble.

Needs `lib.py` and the `.npy` data written by `00_setup_data_feature_importance`. Comments are in Russian; outputs are kept as the record of each experiment.


In [4]:
import os, subprocess, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ GPU')
!df -h /content | tail -1
os.makedirs('/content/data', exist_ok=True)
if not os.path.exists('/content/wnn_connectome_starterpack/datasets/train.parquet'):
    !cd /content && curl -sL https://files.wundernn.io/wnn_connectome_starterpack.tar.gz | tar -xz --wildcards '*/baseline/*' '*/utils.py' '*/datasets/*.parquet' 2>/dev/null
!ls -lh /content/wnn_connectome_starterpack/datasets
!df -h /content | tail -1

GPU: Tesla T4
overlay         113G   85M  113G   1% /
total 29G
-rw-r--r-- 1  501 staff  28G Dec 31  2019 train.parquet
-rw-r--r-- 1  501 staff  88M Dec 31  2019 valid_mask.parquet
-rw------- 1 root root  1.3G Nov 10 15:12 valid.parquet
overlay         113G   29G   85G  26% /


In [5]:
from google.colab import drive
drive.mount('/content/drive')
import os, torch
os.makedirs('/content/drive/MyDrive/wnn', exist_ok=True); os.makedirs('/content/data', exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ GPU: Runtime -> Change runtime type -> T4')

Mounted at /content/drive
GPU: Tesla T4


In [ ]:
import numpy as np, pyarrow.parquet as pq

L, WARM, SAT, U3 = 20000, 99, 99, 0.6569
UN = np.array([0.1254, 0.2413, 0.4685, 0.6569])
ROOT = '/content/wnn_connectome_starterpack/datasets'
post = np.zeros(L, bool); post[WARM:] = True

def fam_of(Y):                                   # семейство = ближайшая единица к наименьшему ненулевому |t0|
    a = np.abs(np.clip(Y[..., 0], -2, 2)).astype(np.float64)
    mn = np.where(a > 0, a, 9.0).min(1); f = np.abs(mn[:, None] - UN[None]).argmin(1)
    print('макс. отклонение от единицы:', np.abs(mn - UN[f]).max().round(4), '| классы:', np.bincount(f, minlength=4)); return f

def code3(y):                                    # код k в единицах семейства 4; насыщение = ±99
    k = np.round(y / U3).astype(np.int32); k[y >= 2 - 1e-6] = SAT; k[y <= -2 + 1e-6] = -SAT; return k

def wc(y, p, m):                                 # взвешенная корреляция по маске m
    y, p = y[m].astype(np.float64), p[m].astype(np.float64); w = np.abs(y); sw = w.sum()
    my, mp = (w * y).sum() / sw, (w * p).sum() / sw
    cov, vy, vp = (w * (y - my) * (p - mp)).sum(), (w * (y - my) ** 2).sum(), (w * (p - mp) ** 2).sum()
    return cov / np.sqrt(vy * vp) if vp > 1e-12 else 0.0

pf = pq.ParquetFile(f'{ROOT}/valid.parquet')
meta = {'seq_ix', 'step_in_seq', 'need_prediction', 'is_scored', 't0', 't1'}
feat = [c for c in pf.schema_arrow.names if c not in meta]; assert len(feat) == 112
idx = np.sort(np.random.default_rng(1).choice(pf.num_row_groups, 300, replace=False))

Yva = np.zeros((300, L, 2), np.float32); Mva = np.zeros((300, L), bool)
for k, i in enumerate(idx):                      # цели и маска для всех 300 (быстро)
    tb = pf.read_row_group(int(i), columns=['t0', 't1', 'is_scored'])
    Yva[k] = np.column_stack([tb['t0'].to_numpy(), tb['t1'].to_numpy()]); Mva[k] = tb['is_scored'].to_numpy()
fam_v = fam_of(Yva); ix3 = np.where(fam_v == 3)[0]          # ждём классы [68 54 80 98]

X3 = np.zeros((len(ix3), L, 112), np.float32)
for r, k in enumerate(ix3):                      # признаки только для F3
    tb = pf.read_row_group(int(idx[k]), columns=feat); X3[r] = np.column_stack([tb[c].to_numpy() for c in feat])

Yc = np.clip(Yva, -2, 2).astype(np.float64)
sel = Mva & np.broadcast_to(post, Mva.shape)
sdX = np.diff(X3, axis=1, prepend=X3[:, :1])
e0, e1 = code3(Yc[ix3, :, 0]) <= -3, code3(Yc[ix3, :, 1]) <= -3
starts = lambda e: np.pad(e[:, 1:] & ~e[:, :-1], ((0, 0), (1, 0)))
on0, on1 = starts(e0), starts(e1); on = on0 | on1
print('F3:', len(ix3), 'последовательностей | начал событий:', int(on[:, WARM + 1:].sum()), '(в прошлый раз 4691)')

макс. отклонение от единицы: 0.0 | классы: [68 54 80 98]
F3: 98 последовательностей | начал событий: 4691 (в прошлый раз 4691)


In [ ]:
# ---------- тест 1: оракул «время с последнего события» (F3) ----------
def since(on):
    out = np.empty(on.shape, np.int64); last = np.full(on.shape[0], -10**7)
    for t in range(on.shape[1]):
        last = np.where(on[:, t], t, last); out[:, t] = t - last
    return out
EDGES = np.array([0, 1, 2, 3, 6, 11, 21, 51, 101, 201, 501, 1001, 10**9]); NB = len(EDGES) - 1
Y3c = Yc[ix3]; s3 = sel[ix3]; n3 = len(ix3); half = np.arange(n3) % 2
Po = np.zeros(Y3c.shape)
for j, on_j in ((0, on0), (1, on1)):
    bins = np.digitize(since(on_j), EDGES) - 1; y = Y3c[..., j]; w = np.abs(y)
    for a in (0, 1):
        tr, te = half != a, half == a; m = np.zeros(bins.shape, bool); m[tr, WARM:] = True
        num = np.bincount(bins[m], weights=(w * y)[m], minlength=NB); den = np.bincount(bins[m], weights=w[m], minlength=NB)
        T = np.where(den > 0, num / np.maximum(den, 1e-9), 0.0); Po[te, :, j] = T[bins[te]]
    if j == 0: print('таблица t0, шаги с события (0,1,2,3-5,6-10,11-20,21-50,51-100,101-200,201-500,501-1000,>1000):\n ', np.round(T, 2))
f3 = lambda P: [round(wc(Y3c[..., j], np.clip(P[..., j], -2, 2).astype(np.float64), s3), 3) for j in (0, 1)]
print('оракул «событие + затухание», F3 (t0, t1):', f3(Po))

# ---------- тест 2: обнаружение по окну t..t+2 ----------
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve
def wf(i, t): return np.concatenate([sdX[i, t], sdX[i, t + 1], sdX[i, t + 2], X3[i, t + 2]], 1)
valid_t = np.arange(WARM + 1, L - 3); rng = np.random.default_rng(0)
Ftr, ytr = [], []
for i in np.where(half == 0)[0]:
    pos = valid_t[on[i, valid_t]]; neg = rng.choice(valid_t[~on[i, valid_t]], 4000, replace=False); t = np.concatenate([pos, neg])
    Ftr.append(wf(i, t)); ytr.append(on[i, t])
Ftr, ytr = np.concatenate(Ftr), np.concatenate(ytr)
clf3 = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.1, random_state=0).fit(Ftr, ytr)
pp, yy = [], []
for i in np.where(half == 1)[0]:
    pp.append(clf3.predict_proba(wf(i, valid_t))[:, 1]); yy.append(on[i, valid_t])
pp, yy = np.concatenate(pp), np.concatenate(yy); pr, rc, _ = precision_recall_curve(yy, pp)
print(f'\nобнаружение по окну t..t+2: AUC {roc_auc_score(yy, pp):.3f} | база {yy.mean():.5f}  (по одному шагу было: полнота 0.5 -> точность 0.112)')
for r in (0.3, 0.5, 0.7, 0.9): print(f'  полнота {r}: точность {pr[np.where(rc >= r)[0].max()]:.3f}')

таблица t0, шаги с события (0,1,2,3-5,6-10,11-20,21-50,51-100,101-200,201-500,501-1000,>1000):
  [-1.97 -1.94 -1.93 -1.86 -1.81 -1.78 -1.68 -1.44 -1.12 -0.87 -0.66 -0.4 ]
оракул «событие + затухание», F3 (t0, t1): [np.float64(0.52), np.float64(0.56)]

обнаружение по окну t..t+2: AUC 0.986 | база 0.00224  (по одному шагу было: полнота 0.5 -> точность 0.112)
  полнота 0.3: точность 0.136
  полнота 0.5: точность 0.122
  полнота 0.7: точность 0.087
  полнота 0.9: точность 0.052


In [ ]:
names = ([f'i0_p{k}' for k in range(22)] + [f'i0_v{k}' for k in range(22)] + [f'i0_dp{k}' for k in range(4)] + [f'i0_dv{k}' for k in range(4)]
         + [f'i1_p{k}' for k in range(22)] + [f'i1_v{k}' for k in range(22)] + [f'i1_dp{k}' for k in range(4)] + [f'i1_dv{k}' for k in range(4)]
         + [f'a{k}' for k in range(8)]); assert len(names) == 112
W = (5, 20, 50, 100, 200, 500, 1000)
rows = np.arange(WARM + 1000, L, 7); Y3c = Yc[ix3]
ones = np.ones(len(ix3) * len(rows), bool)
res = []
for c in range(112):
    x = X3[:, :, c]; cs = np.cumsum(np.abs(sdX[:, :, c]), axis=1, dtype=np.float64)
    feats = {('уровень', 0): x[:, rows]}
    for w in W:
        feats[('смещение', w)] = x[:, rows] - x[:, rows - w]
        feats[('активность', w)] = (cs[:, rows] - cs[:, rows - w]) / w
    for (kind, w), f in feats.items():
        for j in (0, 1):
            r = wc(Y3c[:, rows, j].ravel(), f.ravel(), ones); res.append((abs(r), r, j, kind, names[c], w))
res.sort(reverse=True)
print('топ-20 по |взвешенной корреляции| с таргетом (F3):')
for a, r, j, kind, nm, w in res[:20]: print(f'  t{j} {kind:10s} {nm:8s} окно {w:5d}: {r:+.3f}')
for kind in ('уровень', 'смещение', 'активность'):
    b = max((x for x in res if x[3] == kind), key=lambda x: x[0]); print(f'лучший из «{kind}»: t{b[2]} {b[4]} окно {b[5]}: {b[1]:+.3f}')

топ-20 по |взвешенной корреляции| с таргетом (F3):
  t1 уровень    i0_v0    окно     0: -0.089
  t0 уровень    i0_v19   окно     0: -0.088
  t0 активность i0_dp0   окно   200: +0.085
  t0 активность i0_dp3   окно   200: +0.085
  t0 активность a0       окно   200: +0.084
  t0 активность a0       окно   100: +0.084
  t0 активность i0_dp3   окно   100: +0.084
  t1 уровень    i0_v20   окно     0: -0.083
  t0 активность i0_dp3   окно    50: +0.083
  t0 активность a0       окно    50: +0.082
  t0 активность i0_dp2   окно   200: +0.082
  t0 активность i0_dp1   окно   100: +0.082
  t0 активность i0_dp0   окно   100: +0.081
  t0 активность a6       окно   100: +0.081
  t0 активность i0_dp1   окно   200: +0.081
  t0 активность a6       окно    50: +0.081
  t0 активность i0_p8    окно    20: +0.080
  t0 активность i0_dp1   окно    50: +0.080
  t0 активность i0_p0    окно    20: +0.080
  t0 активность i0_p20   окно    20: +0.080
лучший из «уровень»: t1 i0_v0 окно 0: -0.089
лучший из «смещение»: t1

In [ ]:
G = {'i0 цены': range(0, 22), 'i0 объёмы': range(22, 44), 'i0 сделки': range(44, 52),
     'i1 цены': range(52, 74), 'i1 объёмы': range(74, 96), 'i1 сделки': range(96, 104), 'a0-a7': range(104, 112)}
dX = np.abs(sdX); bgv = dX[:, WARM:].mean((0, 1)) + 1e-9
A = np.stack([(dX[..., list(c)] / bgv[list(c)]).mean(-1) for c in G.values()], -1)    # (98, L, 7); 1.0 = фон
OFF = [-300, -200, -100, -50, -20, -10, -5, -2, 0, 2, 5, 10, 20, 50, 100, 200, 300]

def pick(onm):
    si, ti = np.where(onm); ok = (ti >= WARM + 310) & (ti < L - 310); return si[ok], ti[ok]
def table(si, ti, half=2):
    return np.array([[np.mean([A[si, ti + o + d, g] for d in range(-half, half + 1)]) for o in OFF] for g in range(len(G))])

si, ti = pick(on); T = table(si, ti); rng = np.random.default_rng(0)
dev = max(np.abs(table(rng.integers(0, len(ix3), len(si)), rng.integers(WARM + 310, L - 310, len(si))) - 1).max() for _ in range(5))
print(len(si), 'событий; случайные моменты отклоняются от 1.0 не более чем на', round(float(dev), 2))
print('смещение  ' + ' '.join(f'{o:6d}' for o in OFF))
for g, n in enumerate(G): print(f'{n:10s}' + ' '.join(f'{v:6.2f}' for v in T[g]))

4530 событий; случайные моменты отклоняются от 1.0 не более чем на 0.11
смещение    -300   -200   -100    -50    -20    -10     -5     -2      0      2      5     10     20     50    100    200    300
i0 цены     0.77   0.75   0.78   0.86   0.88   0.98   0.97   1.73   1.94   1.61   0.81   0.62   0.48   0.48   0.50   0.59   0.65
i0 объёмы   0.81   0.80   0.82   0.88   0.90   0.98   0.95   1.71   1.71   1.47   0.81   0.71   0.60   0.60   0.61   0.68   0.73
i0 сделки   1.20   1.17   1.24   1.34   1.27   1.36   1.55   2.44   2.66   2.01   1.01   1.13   1.16   1.31   1.27   1.24   1.19
i1 цены     1.15   1.10   1.15   1.12   1.06   0.96   0.97   0.91   0.63   0.40   0.61   1.03   1.25   1.30   1.34   1.25   1.16
i1 объёмы   1.12   1.10   1.15   1.11   1.05   1.00   0.98   0.90   0.60   0.38   0.64   1.06   1.31   1.34   1.34   1.26   1.18
i1 сделки   1.11   1.06   1.08   1.04   1.00   0.95   0.95   0.90   0.61   0.37   0.55   1.01   1.24   1.32   1.27   1.22   1.14
a0-a7       1.04   1.04  

In [ ]:
from scipy.signal import lfilter
from sklearn.ensemble import HistGradientBoostingRegressor

SP = (20, 100, 300)
def ema_t(Z, span):
    a = 2.0 / (span + 1)
    return lfilter([a], [1, -(1 - a)], Z, axis=1, zi=(1 - a) * Z[:, :1, :])[0]
ACT = []
for s in SP:
    E = ema_t(A, s)                                              # (98, L, 7): 0-2 это i0 цены/объёмы/сделки, 3-5 это i1 те же
    ACT += [E, np.log((E[..., 3:6] + 1e-3) / (E[..., 0:3] + 1e-3))]
ACT = np.concatenate(ACT, -1).astype(np.float32); print('признаков активности:', ACT.shape[-1])

n3 = len(ix3); half = np.arange(n3) % 2; rng = np.random.default_rng(0)
Y3c = Yc[ix3]; s3 = sel[ix3]
sets = {'сырой вход (112)': X3, 'активность (30)': ACT, 'вход + активность (142)': np.concatenate([X3, ACT], -1)}
for name, F in sets.items():
    P = np.zeros(Y3c.shape, np.float32)
    for a in (0, 1):
        tr, te = np.where(half != a)[0], np.where(half == a)[0]
        rows = rng.integers(WARM + 300, L, (len(tr), 3000))
        Xs = F[tr[:, None], rows].reshape(-1, F.shape[-1])
        for j in (0, 1):
            ys = Y3c[tr[:, None], rows, j].ravel()
            m = HistGradientBoostingRegressor(max_iter=150, learning_rate=0.1, early_stopping=False, random_state=0)
            m.fit(Xs, ys, sample_weight=np.abs(ys) + 1e-6)
            P[te, :, j] = m.predict(F[te].reshape(-1, F.shape[-1])).reshape(len(te), L)
    print(f'{name:24s} F3 (t0, t1):', [round(wc(Y3c[..., j], np.clip(P[..., j], -2, 2).astype(np.float64), s3), 3) for j in (0, 1)], flush=True)
print('ориентиры: baseline×0.3 [0.41, 0.388] | GRU [0.406, 0.389] | оракул времени с события [0.52, 0.56]')

признаков активности: 30
сырой вход (112)         F3 (t0, t1): [np.float64(0.184), np.float64(0.164)]
активность (30)          F3 (t0, t1): [np.float64(-0.06), np.float64(0.012)]
вход + активность (142)  F3 (t0, t1): [np.float64(0.203), np.float64(0.213)]
ориентиры: baseline×0.3 [0.41, 0.388] | GRU [0.406, 0.389] | оракул времени с события [0.52, 0.56]


In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, precision_recall_curve

vt = np.arange(WARM + 1, L - 3); y = on[:, vt].ravel(); rng = np.random.default_rng(0)
print('база (доля начал событий):', round(y.mean(), 5), '| ориентир: точность при полноте 0.5 по сырым колонкам 0.112-0.122')

def report(name, s):                                   # s: (98, len(vt))
    s = s.ravel(); pr, rc, _ = precision_recall_curve(y, s)
    print(f'  {name:34s} AUC {roc_auc_score(y, s):.3f} | точность при полноте 0.5: {pr[np.where(rc >= 0.5)[0].max()]:.3f}, при 0.3: {pr[np.where(rc >= 0.3)[0].max()]:.3f}')
def win_max(A): return np.maximum.reduce([A[:, vt + o] for o in (0, 1, 2)])    # максимум по окну t..t+2

groups = {'i0 цены': range(0, 22), 'i0 объёмы': range(22, 44), 'i0 сделки': range(44, 52)}
for gname, cols in groups.items():
    cols = list(cols); Z = X3[:, WARM:, cols].reshape(-1, len(cols))
    pca = PCA(n_components=8).fit(Z[rng.choice(len(Z), 300000, replace=False)])
    print(f'\n{gname}: доля дисперсии PC1..8:', np.round(pca.explained_variance_ratio_, 3))
    D = np.einsum('slc,kc->slk', sdX[..., cols], pca.components_)               # разности в осях главных компонент
    report('сумма |Δ| по колонкам (контроль)', win_max(np.abs(sdX[..., cols]).sum(-1)))
    for k in range(3): report(f'|ΔPC{k + 1}|', win_max(np.abs(D[..., k])))
    report('норма ΔPC1..8', win_max(np.sqrt((D ** 2).sum(-1))))
    del D

a0 = np.abs(sdX[..., :52]).sum(-1); a1 = np.abs(sdX[..., 52:104]).sum(-1)
print('\nкросс-инструментный признак:')
report('i0-вспышка / i1-тишина (t..t+2)', win_max(np.log((a0 + 1e-3) / (a1 + 1e-3))))

база (доля начал событий): 0.0024 | ориентир: точность при полноте 0.5 по сырым колонкам 0.112-0.122

i0 цены: доля дисперсии PC1..8: [0.819 0.105 0.054 0.015 0.004 0.001 0.001 0.   ]
  сумма |Δ| по колонкам (контроль)   AUC 0.773 | точность при полноте 0.5: 0.003, при 0.3: 0.017
  |ΔPC1|                             AUC 0.760 | точность при полноте 0.5: 0.004, при 0.3: 0.007
  |ΔPC2|                             AUC 0.847 | точность при полноте 0.5: 0.007, при 0.3: 0.021
  |ΔPC3|                             AUC 0.788 | точность при полноте 0.5: 0.007, при 0.3: 0.007
  норма ΔPC1..8                      AUC 0.766 | точность при полноте 0.5: 0.003, при 0.3: 0.007

i0 объёмы: доля дисперсии PC1..8: [0.124 0.088 0.084 0.079 0.073 0.066 0.061 0.054]
  сумма |Δ| по колонкам (контроль)   AUC 0.775 | точность при полноте 0.5: 0.005, при 0.3: 0.006
  |ΔPC1|                             AUC 0.760 | точность при полноте 0.5: 0.005, при 0.3: 0.006
  |ΔPC2|                             AUC 0.751 | точ

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
rng = np.random.default_rng(1); te_rows = np.where(half == 1)[0]; Ft, yt = [], []
for i in te_rows:
    pos = valid_t[on[i, valid_t]]; neg = rng.choice(valid_t[~on[i, valid_t]], 3000, replace=False); t = np.concatenate([pos, neg])
    Ft.append(wf(i, t)); yt.append(on[i, t])
Ft, yt = np.concatenate(Ft), np.concatenate(yt)
base_auc, base_ap = roc_auc_score(yt, clf3.predict_proba(Ft)[:, 1]), average_precision_score(yt, clf3.predict_proba(Ft)[:, 1])
print(f'тестовый набор: {len(yt)} строк, из них событий {yt.sum()} ({yt.mean():.3f}) | базовый AUC {base_auc:.3f}, AP {base_ap:.3f}')

BLK = ['Δ на шаге t', 'Δ на шаге t+1', 'Δ на шаге t+2', 'уровень на t+2']
GRP = {'i0 цены': range(0, 22), 'i0 объёмы': range(22, 44), 'i0 сделки': range(44, 52), 'i1 цены': range(52, 74),
       'i1 объёмы': range(74, 96), 'i1 сделки': range(96, 104), 'a0-a7': range(104, 112)}
print('\nпадение AP при перестановке блока x группы колонок (чем больше падение, тем важнее):')
print(' ' * 16 + ''.join(f'{b:>16s}' for b in BLK))
for gname, cols in GRP.items():
    row = []
    for b in range(4):
        idx = [b * 112 + c for c in cols]; Fp = Ft.copy(); Fp[:, idx] = Fp[rng.permutation(len(Fp))][:, idx]
        row.append(base_ap - average_precision_score(yt, clf3.predict_proba(Fp)[:, 1]))
    print(f'{gname:16s}' + ''.join(f'{v:16.3f}' for v in row), flush=True)

тестовый набор: 149182 строк, из них событий 2182 (0.015) | базовый AUC 0.986, AP 0.433

падение AP при перестановке блока x группы колонок (чем больше падение, тем важнее):
                     Δ на шаге t   Δ на шаге t+1   Δ на шаге t+2  уровень на t+2
i0 цены                    0.363           0.010           0.000           0.000
i0 объёмы                  0.002          -0.012           0.023           0.004
i0 сделки                  0.033           0.000           0.001           0.004
i1 цены                    0.001           0.000          -0.000          -0.000
i1 объёмы                  0.000           0.000          -0.000          -0.002
i1 сделки                 -0.000           0.000           0.000           0.002
a0-a7                      0.173           0.013           0.026           0.171


In [ ]:
from sklearn.metrics import average_precision_score
rng = np.random.default_rng(2)
nm = ([f'i0_p{k}' for k in range(22)] + [f'i0_v{k}' for k in range(22)] + [f'i0_dp{k}' for k in range(4)] + [f'i0_dv{k}' for k in range(4)]
      + [f'i1_p{k}' for k in range(22)] + [f'i1_v{k}' for k in range(22)] + [f'i1_dp{k}' for k in range(4)] + [f'i1_dv{k}' for k in range(4)]
      + [f'a{k}' for k in range(8)])
cand = [(0, c) for c in list(range(0, 22)) + list(range(44, 52)) + list(range(104, 112))] + [(3, c) for c in range(104, 112)] + [(2, c) for c in range(22, 44)]
res = []
for b, c in cand:
    Fp = Ft.copy(); Fp[:, b * 112 + c] = Fp[rng.permutation(len(Fp)), b * 112 + c]
    res.append((base_ap - average_precision_score(yt, clf3.predict_proba(Fp)[:, 1]), b, c))
res.sort(reverse=True)
print('топ-12 колонок по падению AP:')
for d, b, c in res[:12]: print(f'  {["Δ на t", "Δ на t+1", "Δ на t+2", "уровень на t+2"][b]:15s} {nm[c]:8s} {d:+.3f}')

ev = yt.astype(bool)
print('\nкак выглядят лучшие колонки на событиях и вне них:')
for d, b, c in res[:5]:
    v = Ft[:, b * 112 + c]
    for lab, m in (('событие', ev), ('не событие', ~ev)):
        x = v[m]
        s = f'  {nm[c]:7s} блок {b} | {lab:10s} | доля нулей {np.mean(x == 0):.2f} | квантили 5/25/50/75/95%: {np.round(np.quantile(x, [.05, .25, .5, .75, .95]), 2)}'
        if b == 3: s += f' | доля заглушки {np.mean(x < -5.19):.2f}'
        print(s)

топ-12 колонок по падению AP:
  Δ на t          i0_p10   +0.213
  Δ на t          a6       +0.163
  уровень на t+2  a5       +0.089
  Δ на t          i0_p14   +0.035
  Δ на t          i0_dp0   +0.030
  уровень на t+2  a2       +0.027
  Δ на t          i0_p20   +0.026
  Δ на t+2        i0_v5    +0.013
  уровень на t+2  a7       +0.009
  уровень на t+2  a4       +0.004
  Δ на t          i0_p15   +0.003
  уровень на t+2  a3       +0.003

как выглядят лучшие колонки на событиях и вне них:
  i0_p10  блок 0 | событие    | доля нулей 0.00 | квантили 5/25/50/75/95%: [-0.41 -0.36 -0.25 -0.23 -0.19]
  i0_p10  блок 0 | не событие | доля нулей 0.69 | квантили 5/25/50/75/95%: [-0.25  0.    0.    0.    0.27]
  a6      блок 0 | событие    | доля нулей 0.54 | квантили 5/25/50/75/95%: [-6.01 -4.54  0.    0.    0.  ]
  a6      блок 0 | не событие | доля нулей 0.11 | квантили 5/25/50/75/95%: [-4.57  0.    0.    0.09  4.04]
  a5      блок 3 | событие    | доля нулей 0.00 | квантили 5/25/50/75/95%: [-5.2  

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import precision_recall_curve, roc_auc_score

CD = {'p10': 10, 'p14': 14, 'p20': 20, 'dp0': 44, 'a6': 110}; CL = {'a5': 109, 'a2': 106, 'a7': 111}
vt = np.arange(WARM + 1, L)
D = np.stack([sdX[:, vt, c] for c in CD.values()] + [X3[:, vt, c] for c in CL.values()], -1)      # (98, L', 8)
fn = [f'Δ{k}' for k in CD] + [f'ур.{k}' for k in CL]
y = on[:, vt]; ev = np.arange(D.shape[0]) % 2
tr, te = D[ev == 0].reshape(-1, 8), D[ev == 1].reshape(-1, 8); ytr, yte = y[ev == 0].ravel(), y[ev == 1].ravel()
print('база на тесте:', round(yte.mean(), 5), '| событий в тесте:', int(yte.sum()))

def pr_at(score, yy, rs=(0.3, 0.5, 0.7)):
    pr, rc, _ = precision_recall_curve(yy, score); return {r: round(float(pr[np.where(rc >= r)[0].max()]), 3) for r in rs}

# 1) ручное правило из таблицы
def rule(Dm, a6=False):
    r = (Dm[:, 0] > -0.45) & (Dm[:, 0] < -0.15) & (Dm[:, 1] > 0.02) & (Dm[:, 1] < 0.32)
    return r & (Dm[:, 4] < -3.5) if a6 else r
for nme, r in (('p10↓ и p14↑', rule(te)), ('то же и a6↓ на 4+', rule(te, True))):
    print(f'правило «{nme}»: срабатывает на {r.mean():.4f} шагов | полнота {yte[r].sum() / yte.sum():.2f} | точность {yte[r].mean():.3f} (база {yte.mean():.4f})')

# 2) дерево на восьми признаках
for depth in (3, 5, 7):
    t = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=100, random_state=0).fit(tr, ytr); p = t.predict_proba(te)[:, 1]
    print(f'дерево глубины {depth}: AUC {roc_auc_score(yte, p):.3f} | точность при полноте {pr_at(p, yte)}')
t = DecisionTreeClassifier(max_depth=4, min_samples_leaf=100, random_state=0).fit(tr, ytr); print(export_text(t, feature_names=fn, decimals=2))

# 3) какая цель ныряет: дерево по разностям на t..t+2 (только настоящие события, ровно одна из двух целей)
one = (on0 ^ on1); si, ti = np.where(one & (np.arange(L)[None, :] >= WARM + 1) & (np.arange(L)[None, :] < L - 3)); ys = on0[si, ti].astype(int)
S = np.concatenate([sdX[si, ti + o] for o in (0, 1, 2)], 1); sn = [f'Δ(t+{o}) {nm[c]}' for o in (0, 1, 2) for c in range(112)]
ts = DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=0).fit(S[si % 2 == 0], ys[si % 2 == 0])
print(f'\nсторона события: точность дерева глубины 3 на тесте {(ts.predict(S[si % 2 == 1]) == ys[si % 2 == 1]).mean():.3f}')
print(export_text(ts, feature_names=sn, decimals=2))

база на тесте: 0.00224 | событий в тесте: 2184
правило «p10↓ и p14↑»: срабатывает на 0.1018 шагов | полнота 0.95 | точность 0.021 (база 0.0022)
правило «то же и a6↓ на 4+»: срабатывает на 0.0147 шагов | полнота 0.42 | точность 0.064 (база 0.0022)
дерево глубины 3: AUC 0.974 | точность при полноте {0.3: 0.116, 0.5: 0.086, 0.7: 0.022}
дерево глубины 5: AUC 0.986 | точность при полноте {0.3: 0.135, 0.5: 0.14, 0.7: 0.099}
дерево глубины 7: AUC 0.986 | точность при полноте {0.3: 0.141, 0.5: 0.135, 0.7: 0.097}
|--- Δp14 <= 0.22
|   |--- Δp20 <= -0.20
|   |   |--- Δdp0 <= 0.14
|   |   |   |--- ур.a5 <= 0.67
|   |   |   |   |--- class: False
|   |   |   |--- ур.a5 >  0.67
|   |   |   |   |--- class: False
|   |   |--- Δdp0 >  0.14
|   |   |   |--- Δa6 <= -4.71
|   |   |   |   |--- class: False
|   |   |   |--- Δa6 >  -4.71
|   |   |   |   |--- class: False
|   |--- Δp20 >  -0.20
|   |   |--- Δp10 <= -0.11
|   |   |   |--- ур.a5 <= -0.84
|   |   |   |   |--- class: False
|   |   |   |--- ур.a5 

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
n = len(ix3); ev = np.arange(n) % 2; rng = np.random.default_rng(0)
vt = np.arange(WARM + 1, L); y = on[:, vt]
D = np.stack([sdX[:, vt, c] for c in (10, 14, 20, 44, 110)] + [X3[:, vt, c] for c in (109, 106, 111)], -1)       # (n, L', 8)

# 1) вероятность события на каждом шаге (кросс-фиттинг по половинам последовательностей)
p = np.zeros((n, L), np.float32)
for a in (0, 1):
    tr, te = np.where(ev != a)[0], np.where(ev == a)[0]; Xtr, ytr = D[tr].reshape(-1, 8), y[tr].ravel()
    idx = np.concatenate([np.where(ytr)[0], rng.choice(np.where(~ytr)[0], 300000, replace=False)])
    c = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.1, random_state=0).fit(Xtr[idx], ytr[idx])
    p[te[:, None], vt[None, :]] = c.predict_proba(D[te].reshape(-1, 8))[:, 1].reshape(len(te), -1)

# 2) сторона по Δp5 на шаге t+1 -> сигнал тревоги по каждой цели, доступный в момент t+1
p5n = np.zeros((n, L), np.float32); p5n[:, :-1] = sdX[:, 1:, 5]
s0 = (p5n > -0.05).astype(np.float32)                                  # 1: ныряет t0, 0: ныряет t1
u = np.zeros((n, 2, L), np.float32); u[:, 0, 1:] = (p * s0)[:, :-1]; u[:, 1, 1:] = (p * (1 - s0))[:, :-1]

# 3) суммы сигнала по лаговым корзинам
LB = [(0, 2), (2, 6), (6, 21), (21, 51), (51, 101), (101, 201), (201, 501), (501, 1001), (1001, 2001)]
C = np.concatenate([np.zeros((n, 2, 1)), np.cumsum(u, -1, dtype=np.float64)], -1); tt = np.arange(L)
FE = np.concatenate([C[..., np.clip(tt - a + 1, 0, L)] - C[..., np.clip(tt - b + 1, 0, L)] for a, b in LB], 1)
FE = np.concatenate([FE.transpose(0, 2, 1), np.ones((n, L, 1))], -1).astype(np.float32)           # (n, L, 19)

# 4) взвешенная регрессия на целях, кросс-фиттинг
Y3c = Yc[ix3]; s3 = sel[ix3]; Pf = np.zeros(Y3c.shape, np.float32)
for j in (0, 1):
    for a in (0, 1):
        tr, te = np.where(ev != a)[0], np.where(ev == a)[0]
        Xf = FE[tr][:, WARM:].reshape(-1, 19).astype(np.float64); yv = Y3c[tr][:, WARM:, j].ravel(); w = np.abs(yv)
        coef = np.linalg.solve(Xf.T @ (Xf * w[:, None]) + 1e-6 * np.eye(19), Xf.T @ (yv * w))
        Pf[te, :, j] = np.clip(FE[te] @ coef, -2, 2)
f3 = lambda P: [round(wc(Y3c[..., j], np.clip(P[..., j], -2, 2).astype(np.float64), s3), 3) for j in (0, 1)]
print('детектор + затухание, F3 (t0, t1):', f3(Pf))
print('ориентиры: baseline×0.3 [0.41, 0.388] | GRU [0.406, 0.389] | оракул событий [0.52, 0.56]')

детектор + затухание, F3 (t0, t1): [np.float64(0.042), np.float64(0.055)]
ориентиры: baseline×0.3 [0.41, 0.388] | GRU [0.406, 0.389] | оракул событий [0.52, 0.56]


In [ ]:
import os
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
for d in ('/content/drive/MyDrive/wnn', '/content'):
    if os.path.exists(f'{d}/Pb_valid300.npy'): Pb = np.load(f'{d}/Pb_valid300.npy'); print('загружено из', d); break
else: raise SystemExit('нет Pb_valid300.npy: нужно пересчитать baseline на F3')
Pbase = Pb[ix3].astype(np.float64) * 0.3                      # F3, масштаб 0.3

def split(P, j):
    y = Yc[ix3][..., j]; p = np.clip(P[..., j], -2, 2).astype(np.float64); m = sel[ix3]
    w = np.abs(y) * m; W = w.sum(); Wi = w.sum(1) + 1e-12
    my_i, mp_i = (w * y).sum(1) / Wi, (w * p).sum(1) / Wi; my, mp = (w * y).sum() / W, (w * p).sum() / W
    cb = (Wi * (my_i - my) * (mp_i - mp)).sum() / W; cw = (w * (y - my_i[:, None]) * (p - mp_i[:, None])).sum() / W
    vyb, vyw = (Wi * (my_i - my) ** 2).sum() / W, (w * (y - my_i[:, None]) ** 2).sum() / W
    vpb, vpw = (Wi * (mp_i - mp) ** 2).sum() / W, (w * (p - mp_i[:, None]) ** 2).sum() / W
    return (round((cb + cw) / np.sqrt((vyb + vyw) * (vpb + vpw)), 3), round(cb / (cb + cw), 2), round(vyb / (vyb + vyw), 2),
            round(cb / np.sqrt(vyb * vpb), 3) if vyb * vpb > 0 else 0, round(cw / np.sqrt(vyw * vpw), 3))

print('t | модель | общая корр. | доля ковариации МЕЖДУ посл. | доля дисперсии таргета между | корр. между | корр. внутри')
for name, P in (('baseline×0.3', Pbase),) + ((('оракул событий', Po),) if 'Po' in globals() else ()):
    for j in (0, 1): print(f't{j} | {name:14s} |', *split(P, j))

загружено из /content/drive/MyDrive/wnn
t | модель | общая корр. | доля ковариации МЕЖДУ посл. | доля дисперсии таргета между | корр. между | корр. внутри
t0 | baseline×0.3   | 0.41 0.12 0.11 0.63 0.394
t1 | baseline×0.3   | 0.388 0.16 0.15 0.6 0.367


In [ ]:
def oracle_delay(k):
    P = np.zeros(Y3c.shape)
    for j, on_j in ((0, on0), (1, on1)):
        sh = np.zeros_like(on_j)
        if k == 0: sh = on_j.copy()
        else: sh[:, k:] = on_j[:, :-k]
        el = since(sh) + (k if k > 0 else 0)
        bins = np.digitize(el, EDGES) - 1; bins = np.clip(bins, 0, NB - 1)
        y = Y3c[..., j]; w = np.abs(y)
        for a in (0, 1):
            tr, te = half != a, half == a; m = np.zeros(bins.shape, bool); m[tr, WARM:] = True
            num = np.bincount(bins[m], weights=(w * y)[m], minlength=NB); den = np.bincount(bins[m], weights=w[m], minlength=NB)
            T = np.where(den > 0, num / np.maximum(den, 1e-9), 0.0); P[te, :, j] = T[bins[te]]
    return P
f3 = lambda P: [round(wc(Y3c[..., j], np.clip(P[..., j], -2, 2).astype(np.float64), s3), 3) for j in (0, 1)]
print('задержка k (шагов) | F3 (t0, t1)   [наши модели: 0.41 / 0.39]')
for k in (0, 5, 10, 20, 50, 100, 200, 500): print(f'{k:6d} | {f3(oracle_delay(k))}')

задержка k (шагов) | F3 (t0, t1)   [наши модели: 0.41 / 0.39]
     0 | [np.float64(0.52), np.float64(0.56)]
     5 | [np.float64(0.497), np.float64(0.537)]
    10 | [np.float64(0.481), np.float64(0.523)]
    20 | [np.float64(0.452), np.float64(0.496)]
    50 | [np.float64(0.372), np.float64(0.422)]
   100 | [np.float64(0.28), np.float64(0.338)]
   200 | [np.float64(0.173), np.float64(0.244)]
   500 | [np.float64(0.068), np.float64(0.135)]


In [ ]:
def decomp(P, j, on_j):
    y = Y3c[..., j]; p = np.clip(P[..., j], -2, 2).astype(np.float64); w = np.abs(y) * s3; W = w.sum()
    my, mp = (w * y).sum() / W, (w * p).sum() / W
    vy, vp = (w * (y - my) ** 2).sum() / W, (w * (p - mp) ** 2).sum() / W
    contrib = w * (y - my) * (p - mp) / W / np.sqrt(vy * vp)          # суммируется в корреляцию
    b = np.clip(np.digitize(since(on_j), EDGES) - 1, 0, NB - 1)
    cc = np.bincount(b.ravel(), weights=contrib.ravel(), minlength=NB)
    rows = np.bincount(b[s3], minlength=NB) / s3.sum()
    wv = np.bincount(b.ravel(), weights=(w * (y - my) ** 2).ravel() / W / vy, minlength=NB)
    return rows, wv, cc
lab = ['0', '1', '2', '3-5', '6-10', '11-20', '21-50', '51-100', '101-200', '201-500', '501-1k', '>1k']
for j, on_j in ((0, on0), (1, on1)):
    r, wv, cb = decomp(Pbase, j, on_j); _, _, co = decomp(Po, j, on_j)
    print(f'\nt{j}: шагов с события | доля строк | доля взвеш. дисперсии | вклад в корреляцию: baseline | оракул | разница')
    for i in range(NB): print(f'{lab[i]:>9s} | {r[i]:.3f} | {wv[i]:.3f} | {cb[i]:+.3f} | {co[i]:+.3f} | {co[i] - cb[i]:+.3f}')
    print(f'{"сумма":>9s} | {r.sum():.3f} | {wv.sum():.3f} | {cb.sum():+.3f} | {co.sum():+.3f}')


t0: шагов с события | доля строк | доля взвеш. дисперсии | вклад в корреляцию: baseline | оракул | разница
        0 | 0.001 | 0.004 | +0.001 | +0.009 | +0.007
        1 | 0.002 | 0.004 | +0.002 | +0.010 | +0.008
        2 | 0.002 | 0.004 | +0.001 | +0.009 | +0.008
      3-5 | 0.004 | 0.008 | +0.003 | +0.018 | +0.015
     6-10 | 0.005 | 0.010 | +0.002 | +0.021 | +0.019
    11-20 | 0.009 | 0.018 | +0.002 | +0.037 | +0.034
    21-50 | 0.032 | 0.050 | +0.016 | +0.085 | +0.069
   51-100 | 0.055 | 0.075 | +0.026 | +0.066 | +0.040
  101-200 | 0.104 | 0.119 | +0.051 | +0.029 | -0.022
  201-500 | 0.216 | 0.184 | +0.087 | -0.001 | -0.088
   501-1k | 0.206 | 0.201 | +0.084 | +0.044 | -0.040
      >1k | 0.364 | 0.321 | +0.133 | +0.194 | +0.061
    сумма | 1.000 | 1.000 | +0.410 | +0.520

t1: шагов с события | доля строк | доля взвеш. дисперсии | вклад в корреляцию: baseline | оракул | разница
        0 | 0.002 | 0.005 | +0.002 | +0.011 | +0.010
        1 | 0.002 | 0.005 | +0.002 | +0.011 | +0.01

In [ ]:
print('F3 (t0, t1); baseline×0.3 один: ', f3(Pbase))
for k in (0, 20, 50):
    Pk = oracle_delay(k)
    for w in (0.3, 0.5, 0.7):
        print(f'задержка {k:3d}, вес оракула {w}:', f3((1 - w) * Pbase + w * Pk))

F3 (t0, t1); baseline×0.3 один:  [np.float64(0.41), np.float64(0.388)]
задержка   0, вес оракула 0.3: [np.float64(0.551), np.float64(0.567)]
задержка   0, вес оракула 0.5: [np.float64(0.603), np.float64(0.619)]
задержка   0, вес оракула 0.7: [np.float64(0.594), np.float64(0.612)]
задержка  20, вес оракула 0.3: [np.float64(0.515), np.float64(0.526)]
задержка  20, вес оракула 0.5: [np.float64(0.555), np.float64(0.568)]
задержка  20, вес оракула 0.7: [np.float64(0.542), np.float64(0.557)]
задержка  50, вес оракула 0.3: [np.float64(0.481), np.float64(0.487)]
задержка  50, вес оракула 0.5: [np.float64(0.51), np.float64(0.519)]
задержка  50, вес оракула 0.7: [np.float64(0.489), np.float64(0.502)]


In [ ]:
import pyarrow.parquet as pq
from sklearn.ensemble import HistGradientBoostingRegressor

pf = pq.ParquetFile(f'{ROOT}/valid.parquet'); N = pf.num_row_groups
N7 = ['i0_p10', 'i0_p14', 'i0_p5', 'a2', 'a5', 'a6', 'a7']                     # порядок 0..6
held = set(np.random.default_rng(1).choice(N, 300, replace=False).tolist())

def ind_feats(Z):                                       # Z: (L, 7); причинные признаки (L, 34)
    d = np.diff(Z, axis=0, prepend=Z[:1]); dp10, dp14, dp5, da6 = d[:, 0], d[:, 1], d[:, 2], d[:, 5]
    c4 = (dp10 > -0.45) & (dp10 < -0.15) & (dp14 > 0.02) & (dp14 < 0.32); c5 = c4 & (da6 < -3.5)
    sh = lambda a: np.concatenate([[False], a[:-1]]); neg = dp5 < -0.05
    ind = np.stack([c4, c5, sh(c4) & neg, sh(c4) & ~neg, sh(c5) & neg, sh(c5) & ~neg], 1).astype(np.float64)
    cs = np.concatenate([np.zeros((1, 6)), np.cumsum(ind, 0)], 0); t = np.arange(len(Z)); out = []
    for W in (10, 30, 100, 300, 1000):
        lo = np.maximum(t - W + 1, 0); out.append((cs[t + 1] - cs[lo]) / (t - lo + 1)[:, None])
    out.append(Z[:, [3, 4, 5, 6]]); return np.concatenate(out, 1).astype(np.float32)

# обучающие F3-последовательности: в valid, но не среди наших 300
rng = np.random.default_rng(0); A, B, nseq = [], [], 0
for i in range(N):
    if i in held: continue
    t0 = np.clip(pf.read_row_group(i, columns=['t0'])['t0'].to_numpy(), -2, 2); a = np.abs(t0); a = a[a > 0]
    if len(a) == 0 or abs(a.min() - 0.6569) > 0.01: continue                       # не F3
    tb = pf.read_row_group(i, columns=N7 + ['t0', 't1']); Z = np.column_stack([tb[c].to_numpy() for c in N7]).astype(np.float64)
    Yt = np.clip(np.column_stack([tb['t0'].to_numpy(), tb['t1'].to_numpy()]), -2, 2)
    f = ind_feats(Z); r = rng.choice(np.arange(WARM, L), 2000, replace=False); A.append(f[r]); B.append(Yt[r]); nseq += 1
A, B = np.concatenate(A), np.concatenate(B); print('обучающих F3-последовательностей:', nseq, '| строк:', A.shape)

models = [HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1, early_stopping=False, random_state=0)
          .fit(A, B[:, j], sample_weight=np.abs(B[:, j]) + 1e-6) for j in (0, 1)]
cols = [10, 14, 5, 106, 109, 110, 111]; Ph = np.zeros(Pbase.shape, np.float32)
for k in range(len(ix3)):
    f = ind_feats(X3[k][:, cols].astype(np.float64))
    for j in (0, 1): Ph[k, :, j] = models[j].predict(f)

Y3c = Yc[ix3]; s3 = sel[ix3]
f3 = lambda P: [round(wc(Y3c[..., j], np.clip(P[..., j], -2, 2).astype(np.float64), s3), 3) for j in (0, 1)]
z = lambda P: P / P[s3].std(0)
print('baseline×0.3 один:', f3(Pbase), '| признаки событий один:', f3(Ph))
for w in (0.2, 0.35, 0.5): print(f'смесь, вес признаков событий {w}:', f3((1 - w) * z(Pbase) + w * z(Ph)))
print('для сравнения, потолок с известными событиями: задержка 20 -> [0.555, 0.568], задержка 50 -> [0.51, 0.519]')

обучающих F3-последовательностей: 599 | строк: (1198000, 34)
baseline×0.3 один: [np.float64(0.41), np.float64(0.388)] | признаки событий один: [np.float64(-0.036), np.float64(-0.007)]
смесь, вес признаков событий 0.2: [np.float64(0.377), np.float64(0.367)]
смесь, вес признаков событий 0.35: [np.float64(0.319), np.float64(0.32)]
смесь, вес признаков событий 0.5: [np.float64(0.198), np.float64(0.202)]
для сравнения, потолок с известными событиями: задержка 20 -> [0.555, 0.568], задержка 50 -> [0.51, 0.519]


In [ ]:
from sklearn.decomposition import PCA
from scipy.signal import lfilter
n = len(ix3); Y3c = Yc[ix3]; s3 = sel[ix3]

def pc1(cols):
    Z = X3[:, WARM:, cols].reshape(-1, len(cols)); pca = PCA(1).fit(Z[::40])
    P = (X3[..., cols].reshape(-1, len(cols)) @ pca.components_[0]).reshape(n, L).astype(np.float64)
    return P, float(pca.explained_variance_ratio_[0])

proxies = {}
for nm_, cols in (('все 22 цены i0', list(range(22))), ('bid 0-10', list(range(11))), ('ask 11-21', list(range(11, 22)))):
    proxies[nm_] = pc1(cols); print(nm_, '| доля дисперсии PC1:', round(proxies[nm_][1], 3))

print('\n1) оракул: корреляция таргета (is_scored, веса |y|) с будущим изменением цены P[t+H] - P[t]; t0 / t1')
Hs = (10, 30, 100, 300, 1000, 3000)
print('прокси'.ljust(16) + ''.join(f'H={h}'.rjust(16) for h in Hs))
for nm_, (P, _) in proxies.items():
    row = []
    for H in Hs:
        d = P[:, H:] - P[:, :-H]
        row.append(' / '.join(f'{wc(Y3c[:, :L - H, j], d, s3[:, :L - H]):+.2f}' for j in (0, 1)))
    print(nm_.ljust(16) + ''.join(r.rjust(16) for r in row))

print('\n2) причинный признак: P[t] - EMA(P, span); t0 / t1')
Ss = (20, 100, 500, 2000, 5000)
print('прокси'.ljust(16) + ''.join(f'span={s}'.rjust(16) for s in Ss))
for nm_, (P, _) in proxies.items():
    row = []
    for s in Ss:
        a = 2.0 / (s + 1); E = lfilter([a], [1, -(1 - a)], P, axis=1, zi=(1 - a) * P[:, :1])[0]
        row.append(' / '.join(f'{wc(Y3c[..., j], P - E, s3):+.2f}' for j in (0, 1)))
    print(nm_.ljust(16) + ''.join(r.rjust(16) for r in row))

все 22 цены i0 | доля дисперсии PC1: 0.87
bid 0-10 | доля дисперсии PC1: 0.895
ask 11-21 | доля дисперсии PC1: 0.851

1) оракул: корреляция таргета (is_scored, веса |y|) с будущим изменением цены P[t+H] - P[t]; t0 / t1
прокси                      H=10            H=30           H=100           H=300          H=1000          H=3000
все 22 цены i0     -0.03 / +0.01   -0.02 / +0.00   -0.02 / +0.01   +0.00 / +0.01   -0.02 / +0.02   -0.02 / +0.01
bid 0-10           -0.03 / +0.01   -0.02 / +0.00   -0.02 / +0.01   +0.00 / +0.01   -0.01 / +0.02   -0.02 / +0.01
ask 11-21          +0.04 / -0.02   +0.03 / -0.01   +0.03 / -0.02   -0.00 / -0.01   +0.02 / -0.02   +0.03 / -0.01

2) причинный признак: P[t] - EMA(P, span); t0 / t1
прокси                   span=20        span=100        span=500       span=2000       span=5000
все 22 цены i0     +0.02 / +0.01   +0.02 / -0.00   +0.03 / -0.01   +0.05 / -0.02   +0.08 / -0.03
bid 0-10           +0.01 / +0.01   +0.02 / -0.00   +0.03 / -0.01   +0.05 / -0.02   

In [ ]:
%run -i /content/drive/MyDrive/wnn/lib.py

need = {'train': (600, 0, False), 'train3k': (3000, 0, False), 'valid300': (300, 1, True)}
def done(n): return all(os.path.exists(f'/content/data/{n}_{s}.npy') for s in (['X', 'Y'] + (['M'] if n.startswith('valid') else [])))
missing = [n for n in need if not done(n)]
print('нет файлов для:', missing or 'всё на месте')
if missing:
    if not os.path.exists(f'{ROOT}/train.parquet'):
        print('качаю данные (около 11 минут)...')
        subprocess.run("curl -sL https://files.wundernn.io/wnn_connectome_starterpack.tar.gz | "
                       "tar -xz --wildcards '*/baseline/*' '*/utils.py' '*/datasets/*.parquet' 2>/dev/null",
                       shell=True, cwd='/content')
    for n in missing:
        src = 'valid' if n.startswith('valid') else 'train'
        build(src, n, *need[n])
print(subprocess.run('df -h /content | tail -1', shell=True, capture_output=True, text=True).stdout)

нет файлов для: всё на месте
overlay         113G   66G   47G  59% /



In [ ]:
%run -i /content/drive/MyDrive/wnn/lib.py
ld = lambda n: np.load(f'/content/data/{n}.npy', mmap_mode='r')
Xtr = ld('train_X'); Ytr = np.load('/content/data/train_Y.npy')
Xva = ld('valid300_X'); Yva = np.load('/content/data/valid300_Y.npy'); Mva = np.load('/content/data/valid300_M.npy')
assert Xtr.shape[-1] == 112 and Xva.shape[-1] == 112 and len(Xtr) == 600
fam_v = fam_of(Yva, verbose=False); have = np.bincount(fam_v, minlength=4) / len(fam_v)

def make_beta_loss(beta):
    def f(p, y, mask):
        y = y.clamp(-2, 2); w = y.abs().pow(beta)
        return (w * (p - y) ** 2)[mask].mean()
    return f

for beta in (0.0, 0.5, 1.5):
    loss_fn = make_beta_loss(beta)
    for seed in (0, 1):
        torch.manual_seed(seed); m = train(Net().to(dev), epochs=8, tag=f'beta_{seed}')
        print('>>> вес |y|^', beta, 'сид', seed, mix_score(m), '| контроль (вес |y|^1): 0.467/0.640 и 0.469/0.641', flush=True)
loss_fn = lambda p, y, mask: (y.clamp(-2, 2).abs() * (p - y.clamp(-2, 2)) ** 2)[mask].mean()

/content/drive/MyDrive/wnn/lib.py:75: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  x = torch.from_numpy(np.ascontiguousarray(X[s:s + bs])).to(dev)


эпоха 1/8 | 41 с | loss 0.7642 | valid все строки 0.422 [0.418 0.426] | is_scored 0.629 [0.62  0.637]
эпоха 2/8 | 38 с | loss 0.7251 | valid все строки 0.435 [0.446 0.424] | is_scored 0.627 [0.632 0.623]
эпоха 3/8 | 38 с | loss 0.7116 | valid все строки 0.441 [0.45  0.433] | is_scored 0.631 [0.631 0.631]
эпоха 4/8 | 37 с | loss 0.7083 | valid все строки 0.440 [0.426 0.453] | is_scored 0.634 [0.623 0.645]
эпоха 5/8 | 35 с | loss 0.6970 | valid все строки 0.444 [0.435 0.454] | is_scored 0.630 [0.618 0.642]
эпоха 6/8 | 34 с | loss 0.6903 | valid все строки 0.453 [0.455 0.45 ] | is_scored 0.645 [0.64  0.649]
эпоха 7/8 | 33 с | loss 0.6772 | valid все строки 0.456 [0.455 0.457] | is_scored 0.645 [0.639 0.65 ]
эпоха 8/8 | 32 с | loss 0.6759 | valid все строки 0.457 [0.455 0.459] | is_scored 0.647 [0.641 0.653]
>>> вес |y|^ 0.0 сид 0 {'все строки': 0.455, 'is_scored': 0.635} | контроль (вес |y|^1): 0.467/0.640 и 0.469/0.641
эпоха 1/8 | 33 с | loss 0.7627 | valid все строки 0.421 [0.413 0.428]

In [ ]:
import torch.nn.functional as nnF
EDG = np.array([0, 1, 2, 3, 6, 11, 21, 51, 101, 201, 501, 1001])
def since_labels(Y):
    y = np.clip(Y, -2, 2); lab = np.zeros(y.shape, np.uint8)
    for j in (0, 1):
        e = y[..., j] <= -1.4; st = np.zeros_like(e); st[:, 1:] = e[:, 1:] & ~e[:, :-1]; st[:, 0] = e[:, 0]
        last = np.full(len(y), -1)
        for t in range(y.shape[1]):
            last = np.where(st[:, t], t, last)
            b = np.clip(np.searchsorted(EDG, t - last, side='right') - 1, 0, 11)
            lab[:, t, j] = np.where(last < 0, 12, b)
    return lab
LAB = since_labels(Ytr); print('доли классов t0:', np.round(np.bincount(LAB[..., 0].ravel(), minlength=13) / LAB[..., 0].size, 3))

class NetEv(nn.Module):
    def __init__(s, h=128):
        super().__init__(); s.gru = nn.GRU(NF, h, num_layers=2, batch_first=True); s.head = nn.Linear(h, 2 + 26)
    def forward(s, x, h=None):
        y, h = s.gru(x, h); o = s.head(y)
        return (o if s.training else o[..., :2]), h

def train_ev(model, epochs=8, bs=32, ch=1000, lr=2e-3, lam=0.1):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * (len(Xtr) // bs) * (L // ch), pct_start=0.05)
    rng = np.random.default_rng(0)
    for ep in range(epochs):
        order = rng.permutation(len(Xtr)); rm, rc = [], []
        for g in range(len(Xtr) // bs):
            idx = np.sort(order[g * bs:(g + 1) * bs])
            x = torch.from_numpy(np.ascontiguousarray(Xtr[idx])).to(dev); y = torch.from_numpy(Ytr[idx]).to(dev).clamp(-2, 2)
            lb = torch.from_numpy(LAB[idx]).to(dev).long(); h = None
            for t in range(0, L, ch):
                o, h = model(x[:, t:t + ch], h); h = h.detach()
                mask = (torch.arange(t, t + ch, device=dev) >= WARM)[None].expand(len(idx), -1); yy = y[:, t:t + ch]
                main = (yy.abs() * (o[..., :2] - yy) ** 2)[mask].mean()
                lg = o[..., 2:].reshape(len(idx), -1, 2, 13)[mask]; ce = nnF.cross_entropy(lg.reshape(-1, 13), lb[:, t:t + ch][mask].reshape(-1))
                loss = main + lam * ce; opt.zero_grad(set_to_none=True); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); rm.append(main.item()); rc.append(ce.item())
        print(f'эпоха {ep + 1}/{epochs} | MSE-часть {np.mean(rm):.4f} | CE событий {np.mean(rc):.3f}', flush=True)
    return model

for seed in (0, 1):
    torch.manual_seed(seed); m = train_ev(NetEv().to(dev))
    print('>>> вспомогательная цель про события, сид', seed, mix_score(m), '| контроль: 0.467/0.640 и 0.469/0.641', flush=True)

доли классов t0: [0.003 0.003 0.003 0.008 0.011 0.019 0.049 0.071 0.116 0.221 0.178 0.267
 0.052]
эпоха 1/8 | MSE-часть 1.1478 | CE событий 1.959
эпоха 2/8 | MSE-часть 1.0590 | CE событий 1.715
эпоха 3/8 | MSE-часть 1.0313 | CE событий 1.637
эпоха 4/8 | MSE-часть 1.0210 | CE событий 1.564
эпоха 5/8 | MSE-часть 0.9948 | CE событий 1.488
эпоха 6/8 | MSE-часть 0.9780 | CE событий 1.437
эпоха 7/8 | MSE-часть 0.9528 | CE событий 1.398
эпоха 8/8 | MSE-часть 0.9484 | CE событий 1.378
>>> вспомогательная цель про события, сид 0 {'все строки': 0.47, 'is_scored': 0.64} | контроль: 0.467/0.640 и 0.469/0.641
эпоха 1/8 | MSE-часть 1.1403 | CE событий 1.974
эпоха 2/8 | MSE-часть 1.0573 | CE событий 1.723
эпоха 3/8 | MSE-часть 1.0311 | CE событий 1.645
эпоха 4/8 | MSE-часть 1.0199 | CE событий 1.572
эпоха 5/8 | MSE-часть 0.9941 | CE событий 1.506
эпоха 6/8 | MSE-часть 0.9763 | CE событий 1.450
эпоха 7/8 | MSE-часть 0.9521 | CE событий 1.421
эпоха 8/8 | MSE-часть 0.9462 | CE событий 1.399
>>> вспомога

In [ ]:
def make_beta_loss(beta):
    def f(p, y, mask):
        y = y.clamp(-2, 2); return (y.abs().pow(beta) * (p - y) ** 2)[mask].mean()
    return f
for beta in (2.0, 3.0):
    loss_fn = make_beta_loss(beta)
    for seed in (0, 1):
        torch.manual_seed(seed); m = train(Net().to(dev), epochs=8, tag=f'beta_{seed}')
        print('>>> вес |y|^', beta, 'сид', seed, mix_score(m), '| контроль β=1: 0.467/0.640 и 0.469/0.641 | β=1.5: 0.471/0.644 и 0.469/0.642', flush=True)
loss_fn = make_beta_loss(1.0)

эпоха 1/8 | 35 с | loss 1.9404 | valid все строки 0.426 [0.428 0.424] | is_scored 0.615 [0.599 0.631]
эпоха 2/8 | 34 с | loss 1.7629 | valid все строки 0.433 [0.435 0.431] | is_scored 0.597 [0.593 0.601]
эпоха 3/8 | 34 с | loss 1.7053 | valid все строки 0.457 [0.464 0.45 ] | is_scored 0.637 [0.637 0.636]
эпоха 4/8 | 34 с | loss 1.6843 | valid все строки 0.447 [0.447 0.448] | is_scored 0.624 [0.624 0.624]
эпоха 5/8 | 37 с | loss 1.6310 | valid все строки 0.466 [0.471 0.462] | is_scored 0.641 [0.648 0.634]
эпоха 6/8 | 35 с | loss 1.5938 | valid все строки 0.468 [0.472 0.464] | is_scored 0.652 [0.643 0.66 ]
эпоха 7/8 | 35 с | loss 1.5413 | valid все строки 0.471 [0.472 0.469] | is_scored 0.653 [0.649 0.657]
эпоха 8/8 | 35 с | loss 1.5322 | valid все строки 0.470 [0.471 0.469] | is_scored 0.655 [0.649 0.661]
>>> вес |y|^ 2.0 сид 0 {'все строки': 0.471, 'is_scored': 0.642} | контроль β=1: 0.467/0.640 и 0.469/0.641 | β=1.5: 0.471/0.644 и 0.469/0.642
эпоха 1/8 | 35 с | loss 1.9310 | valid все

In [ ]:
PI = list(range(0, 22)) + list(range(44, 48)) + list(range(52, 74)) + list(range(96, 100))
VI = list(range(22, 44)) + list(range(48, 52)) + list(range(74, 96)) + list(range(100, 104))
AI = list(range(104, 112))
class NetTwo(nn.Module):
    def __init__(s, h=96):
        super().__init__(); s.ip, s.iv = PI + AI, VI + AI
        s.g1 = nn.GRU(len(s.ip), h, num_layers=2, batch_first=True)
        s.g2 = nn.GRU(len(s.iv), h, num_layers=2, batch_first=True); s.head = nn.Linear(2 * h, 2)
    def forward(s, x, h=None):
        h1, h2 = (None, None) if h is None else (h[:2].contiguous(), h[2:].contiguous())
        y1, h1 = s.g1(x[..., s.ip], h1); y2, h2 = s.g2(x[..., s.iv], h2)
        return s.head(torch.cat([y1, y2], -1)), torch.cat([h1, h2], 0)
loss_fn = make_beta_loss(1.0)
for seed in (0, 1):
    torch.manual_seed(seed); m = train(NetTwo().to(dev), epochs=8, tag=f'two_{seed}')
    print('>>> две ветки, параметров', sum(p.numel() for p in m.parameters()), 'сид', seed, mix_score(m), '| контроль: 0.467/0.640 и 0.469/0.641', flush=True)

эпоха 1/8 | 37 с | loss 1.1439 | valid все строки 0.426 [0.426 0.426] | is_scored 0.616 [0.613 0.619]
эпоха 2/8 | 42 с | loss 1.0654 | valid все строки 0.446 [0.45  0.443] | is_scored 0.616 [0.619 0.612]
эпоха 3/8 | 39 с | loss 1.0356 | valid все строки 0.458 [0.46  0.457] | is_scored 0.629 [0.625 0.634]
эпоха 4/8 | 37 с | loss 1.0173 | valid все строки 0.460 [0.456 0.464] | is_scored 0.639 [0.634 0.643]
эпоха 5/8 | 38 с | loss 0.9915 | valid все строки 0.467 [0.463 0.472] | is_scored 0.649 [0.642 0.656]
эпоха 6/8 | 40 с | loss 0.9723 | valid все строки 0.466 [0.467 0.465] | is_scored 0.650 [0.641 0.659]
эпоха 7/8 | 38 с | loss 0.9476 | valid все строки 0.468 [0.465 0.47 ] | is_scored 0.651 [0.643 0.659]
эпоха 8/8 | 39 с | loss 0.9443 | valid все строки 0.467 [0.464 0.469] | is_scored 0.652 [0.643 0.66 ]
>>> две ветки, параметров 203138 сид 0 {'все строки': 0.467, 'is_scored': 0.639} | контроль: 0.467/0.640 и 0.469/0.641
эпоха 1/8 | 38 с | loss 1.1455 | valid все строки 0.429 [0.432 0.

In [ ]:
%run -i /content/drive/MyDrive/wnn/lib.py
ld = lambda n: np.load(f'/content/data/{n}.npy', mmap_mode='r')
Xtr = ld('train_X'); Ytr = np.load('/content/data/train_Y.npy')
Xva = ld('valid300_X'); Yva = np.load('/content/data/valid300_Y.npy'); Mva = np.load('/content/data/valid300_M.npy')
assert Xtr.shape[-1] == 112 and Xva.shape[-1] == 112 and len(Xtr) == 600
fam_v = fam_of(Yva, verbose=False); have = np.bincount(fam_v, minlength=4) / len(fam_v)

def make_beta_loss(beta):
    def f(p, y, mask):
        y = y.clamp(-2, 2); w = y.abs().pow(beta)
        return (w * (p - y) ** 2)[mask].mean()
    return f


from scipy.signal import lfilter
class EmaNorm:
    def __init__(s, X, span): s.X, s.a = X, 2.0 / (span + 1)
    def __len__(s): return len(s.X)
    def __getitem__(s, i):
        x = np.asarray(s.X[i], np.float64)
        e = lfilter([s.a], [1, -(1 - s.a)], x, axis=1, zi=(1 - s.a) * x[:, :1])[0]
        return (x - e).astype(np.float32)

Xtr0, Xva0 = Xtr, Xva
loss_fn = make_beta_loss(1.0)
for span in (500, 5000):
    Xtr, Xva = EmaNorm(Xtr0, span), EmaNorm(Xva0, span)
    for seed in (0, 1):
        torch.manual_seed(seed); m = train(Net().to(dev), epochs=8, tag=f'ema_{seed}')
        print('>>> вход минус EMA, span', span, 'сид', seed, mix_score(m), '| контроль: 0.467/0.640 и 0.469/0.641', flush=True)
Xtr, Xva = Xtr0, Xva0

эпоха 1/8 | 106 с | loss 1.1504 | valid все строки 0.430 [0.434 0.425] | is_scored 0.637 [0.628 0.645]
эпоха 2/8 | 99 с | loss 1.0641 | valid все строки 0.442 [0.45  0.434] | is_scored 0.644 [0.637 0.652]
эпоха 3/8 | 98 с | loss 1.0281 | valid все строки 0.450 [0.454 0.447] | is_scored 0.649 [0.645 0.652]
эпоха 4/8 | 97 с | loss 1.0152 | valid все строки 0.448 [0.443 0.454] | is_scored 0.645 [0.641 0.65 ]
эпоха 5/8 | 97 с | loss 0.9819 | valid все строки 0.457 [0.456 0.458] | is_scored 0.650 [0.641 0.659]
эпоха 6/8 | 97 с | loss 0.9445 | valid все строки 0.439 [0.451 0.427] | is_scored 0.646 [0.64  0.652]
эпоха 7/8 | 97 с | loss 0.8955 | valid все строки 0.431 [0.438 0.425] | is_scored 0.639 [0.634 0.645]
эпоха 8/8 | 96 с | loss 0.8738 | valid все строки 0.434 [0.438 0.429] | is_scored 0.641 [0.635 0.647]
>>> вход минус EMA, span 500 сид 0 {'все строки': 0.434, 'is_scored': 0.629} | контроль: 0.467/0.640 и 0.469/0.641
эпоха 1/8 | 88 с | loss 1.1444 | valid все строки 0.430 [0.434 0.425

In [5]:
import numpy as np
w = np.load('/content/drive/MyDrive/wnn/weights.npz')
for k in w.files: print(k, w[k].shape, w[k].dtype)
print('mods:', w['mods']); print('coef:', w['coef']); print('const:', w['const'])

flat (3698,) float32
mats (411008,) uint16
meta (26, 3) int64
mods (2, 4) int64
coef (2, 2) float32
const (2,) float32
selftest_x (8, 112) float32
selftest_y (8, 2) float32
mods: [[  0   0   0 128]
 [  1  10 256 128]]
coef: [[0.37173402 0.34923056]
 [0.3733302  0.3582178 ]]
const: [0.22734952 0.15965948]


In [16]:
import numpy as np, torch, os, time, shutil, importlib.util
from types import SimpleNamespace
torch.set_num_threads(2)
SRC, OUT, L, WARM, NS = '/content/ens_src', '/content/ens_new', 20000, 99, 20
CK = {'gru_6573': '/content/drive/MyDrive/wnn/gru_full_val_best.pt', 'gru_5000': '/content/drive/MyDrive/wnn/gru_full_best.pt',
      'corr_3k': '/content/drive/MyDrive/wnn/corr3k_1_best.pt'}
CK = {k: v for k, v in CK.items() if os.path.exists(v)}; print('подключаем:', list(CK)); assert CK
XV = np.load('/content/data/valid300_X.npy', mmap_mode='r'); assert XV.shape[-1] == 112

def load_pm(d, name):
    sp = importlib.util.spec_from_file_location(name, f'{d}/solution.py'); m = importlib.util.module_from_spec(sp); sp.loader.exec_module(m); return m.PredictionModel()
def run_pm(pm, X, sid):
    out = np.zeros((len(X), 2), np.float32)
    for t in range(len(X)): out[t] = pm.predict(SimpleNamespace(seq_ix=sid, state=X[t], need_prediction=True))
    return out

class RefNet(torch.nn.Module):
    def __init__(s, h=128):
        super().__init__(); s.gru = torch.nn.GRU(112, h, num_layers=2, batch_first=True); s.head = torch.nn.Linear(h, 2)
    def forward(s, x): return s.head(s.gru(x)[0])
def torch_out(net, X):
    with torch.no_grad(): return net(torch.from_numpy(np.ascontiguousarray(X, dtype=np.float32))).numpy()
nets = {}
for k, p in CK.items():
    n = RefNet(); n.load_state_dict(torch.load(p, map_location='cpu')); nets[k] = n.eval()

W0 = dict(np.load(f'{SRC}/weights.npz')); pm_old = load_pm(SRC, 'sol_old')
Xs = np.asarray(XV[:NS], np.float32); t = time.time()
old = np.stack([run_pm(pm_old, Xs[i], i) for i in range(NS)]).astype(np.float64); print(f'старый ансамбль на {NS} посл.: {time.time() - t:.0f} с')
old_nc = old - W0['const'].astype(np.float64)
P = {k: np.concatenate([torch_out(n, Xs[i:i + 5]) for i in range(0, NS, 5)]).astype(np.float64) for k, n in nets.items()}
st = lambda a: (a[:, WARM:].reshape(-1, 2).mean(0), a[:, WARM:].reshape(-1, 2).std(0))
mu_o, sd_o = st(old_nc); m_old = st(old)[0]; mk = {k: st(a) for k, a in P.items()}
U = 2 * (old_nc - mu_o) / sd_o + sum((P[k] - mk[k][0]) / mk[k][1] for k in P)
s = sd_o / st(U)[1]; A = 2 * s / sd_o; w = {k: s / mk[k][1] for k in P}
const_new = m_old - A * mu_o - sum(w[k] * mk[k][0] for k in P)
print('масштаб старой части A:', A.round(3), '| коэффициенты новых:', {k: w[k].round(3) for k in w}, '| const:', const_new.round(3))

def append_model(Wd, sd):
    mats, flat, meta, mods = Wd['mats'], Wd['flat'], Wd['meta'], Wd['mods']
    mo, fo, r = len(mats), len(flat), len(meta); so = int((mods[:, 2] + 2 * mods[:, 3]).max()); mc, fc, rows = [], [], []
    def addm(a):
        nonlocal mo
        a = np.ascontiguousarray(a, dtype=np.float16); mc.append(a.view(np.uint16).reshape(-1)); rows.append([mo, a.shape[0], a.shape[1]]); mo += a.size
    def addv(v):
        nonlocal fo
        v = np.ascontiguousarray(v, dtype=np.float32); fc.append(v); rows.append([fo, 1, len(v)]); fo += len(v)
    g = lambda key: sd[key].numpy().astype(np.float32)
    addm(g('gru.weight_ih_l0').T); addm(g('gru.weight_hh_l0').T); addv(g('gru.bias_ih_l0')); addv(g('gru.bias_hh_l0'))
    addm(g('gru.weight_ih_l1').T); addm(g('gru.weight_hh_l1').T); addv(g('gru.bias_ih_l1')); addv(g('gru.bias_hh_l1'))
    addm(g('head.weight').T); addv(g('head.bias'))
    Wd['mats'] = np.concatenate([mats] + mc); Wd['flat'] = np.concatenate([flat] + fc)
    Wd['meta'] = np.vstack([meta, np.array(rows, np.int64)]); Wd['mods'] = np.vstack([mods, np.array([[0, r, so, 128]], np.int64)])

Wn = dict(W0)
for k in nets: append_model(Wn, nets[k].state_dict())
ptest = {k: torch_out(nets[k], W0['selftest_x'][None])[0].astype(np.float64) for k in nets}
Wn['coef'] = np.vstack([A * W0['coef'], np.stack([w[k] for k in nets])]).astype(np.float32)
Wn['const'] = const_new.astype(np.float32)
Wn['selftest_y'] = (A * (W0['selftest_y'].astype(np.float64) - W0['const']) + sum(w[k] * ptest[k] for k in nets) + const_new).astype(np.float32)
os.makedirs(OUT, exist_ok=True); np.savez(f'{OUT}/weights.npz', **Wn); shutil.copy(f'{SRC}/solution.py', OUT)
print('записано, моделей в смеси:', len(Wn['mods']))

подключаем: ['gru_6573', 'gru_5000', 'corr_3k']
старый ансамбль на 20 посл.: 29 с


/tmp/ipykernel_513/2845434042.py:22: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  with torch.no_grad(): return net(torch.from_numpy(np.ascontiguousarray(X, dtype=np.float32))).numpy()


масштаб старой части A: [0.409 0.409] | коэффициенты новых: {'gru_6573': array([0.15 , 0.144]), 'gru_5000': array([0.151, 0.146]), 'corr_3k': array([0.15 , 0.144])} | const: [0.23 0.18]
записано, моделей в смеси: 5


In [13]:
!mkdir -p /content/ens_src
!cp /content/drive/MyDrive/wnn/weights.npz /content/ens_src/weights.npz
!ls -lh /content/ens_src/

total 824K
-rw------- 1 root root 824K Nov 10 10:58 weights.npz


In [15]:
!cp /content/drive/MyDrive/wnn/solution.py /content/ens_src/solution.py

In [17]:
pm_new = load_pm(OUT, 'sol_new'); print('самопроверка пройдена')
Xe = np.asarray(XV[20:22], np.float32); worst = 0.0
for i in range(2):
    o_old = run_pm(pm_old, Xe[i], 100 + i).astype(np.float64); o_new = run_pm(pm_new, Xe[i], 100 + i).astype(np.float64)
    exp = A * (o_old - W0['const']) + sum(w[k] * torch_out(nets[k], Xe[i:i + 1])[0] for k in nets) + const_new
    worst = max(worst, np.abs(o_new - exp).max())
print('макс. расхождение с формулой:', worst, '(нужно < 1e-3)')
t = time.time(); run_pm(pm_old, Xe[0], 200); t_old = time.time() - t
t = time.time(); run_pm(pm_new, Xe[0], 201); t_new = time.time() - t
print(f'на строку: старый {t_old / L * 1e6:.0f} мкс, новый {t_new / L * 1e6:.0f} мкс | оценка на платформе ≈ {18.4 * t_new / t_old:.0f} мин (метод грубый, по отношению к 18m22s друга)')
shutil.make_archive('/content/ens_new_submission', 'zip', OUT); shutil.copy('/content/ens_new_submission.zip', '/content/drive/MyDrive/wnn/')

самопроверка пройдена
макс. расхождение с формулой: 0.00014971984800232763 (нужно < 1e-3)
на строку: старый 64 мкс, новый 163 мкс | оценка на платформе ≈ 47 мин (метод грубый, по отношению к 18m22s друга)


'/content/drive/MyDrive/wnn/ens_new_submission.zip'

In [18]:
import numpy as np, torch, os, time, shutil, importlib.util
from types import SimpleNamespace
torch.set_num_threads(2)

# ---------------- настройки ----------------
SRC = '/content/ens_src'                                    # solution.py и weights.npz друга
OUT = '/content/ens3'                                       # сюда соберём новый архив
CKPT = '/content/drive/MyDrive/wnn/gru_full_val_best.pt'    # наша GRU 2x128, обучена на 6573 последовательностях
XV_PATH = '/content/data/valid300_X.npy'
WN = 1.0                                                    # вес нашей модели относительно пары друга (1.0 = все три поровну)
L, WARM, NS = 20000, 99, 20
for p in (f'{SRC}/solution.py', f'{SRC}/weights.npz', CKPT, XV_PATH):
    assert os.path.exists(p), f'нет файла: {p}'
XV = np.load(XV_PATH, mmap_mode='r'); assert XV.shape[-1] == 112 and len(XV) >= 22

# ---------------- вспомогательное ----------------
def load_pm(d, name):
    sp = importlib.util.spec_from_file_location(name, f'{d}/solution.py'); m = importlib.util.module_from_spec(sp); sp.loader.exec_module(m)
    return m.PredictionModel()
def run_pm(pm, X, sid):
    out = np.zeros((len(X), 2), np.float32)
    for t in range(len(X)): out[t] = pm.predict(SimpleNamespace(seq_ix=sid, state=X[t], need_prediction=True))
    return out
class RefNet(torch.nn.Module):
    def __init__(s, h):
        super().__init__(); s.gru = torch.nn.GRU(112, h, num_layers=2, batch_first=True); s.head = torch.nn.Linear(h, 2)
    def forward(s, x): return s.head(s.gru(x)[0])
def torch_out(net, X):
    with torch.no_grad(): return net(torch.from_numpy(np.ascontiguousarray(X, dtype=np.float32))).numpy()

sd = torch.load(CKPT, map_location='cpu'); H = int(sd['gru.weight_hh_l0'].shape[1])
net = RefNet(H); net.load_state_dict(sd); net.eval(); print('наша модель: скрытое состояние', H)

# ---------------- статистики старого ансамбля и нашей модели на 20 последовательностях ----------------
W0 = dict(np.load(f'{SRC}/weights.npz')); assert W0['mats'].dtype == np.uint16
pm_old = load_pm(SRC, 'sol_old')
Xs = np.asarray(XV[:NS], np.float32); t = time.time()
old = np.stack([run_pm(pm_old, Xs[i], i) for i in range(NS)]).astype(np.float64); print(f'старый ансамбль на {NS} посл.: {time.time() - t:.0f} с')
old_nc = old - W0['const'].astype(np.float64)
P = np.concatenate([torch_out(net, Xs[i:i + 5]) for i in range(0, NS, 5)]).astype(np.float64)
st = lambda a: (a[:, WARM:].reshape(-1, 2).mean(0), a[:, WARM:].reshape(-1, 2).std(0))
mu_o, sd_o = st(old_nc); m_old = st(old)[0]; mu_p, sd_p = st(P)

# смесь: старая пара считается за две модели, наша за WN; итог сохраняет масштаб и центр старого ансамбля
U = 2 * (old_nc - mu_o) / sd_o + WN * (P - mu_p) / sd_p
s = sd_o / st(U)[1]; A = 2 * s / sd_o; w = WN * s / sd_p
const_new = m_old - A * mu_o - w * mu_p
print('множитель старой части:', A.round(3), '| коэффициент нашей модели:', w.round(3), '| const:', const_new.round(3))

# ---------------- дописываем нашу модель как модель типа 0 ----------------
mats, flat, meta, mods = W0['mats'], W0['flat'], W0['meta'], W0['mods']
mo, fo, r = len(mats), len(flat), len(meta); so = int((mods[:, 2] + 2 * mods[:, 3]).max()); mc, fc, rows = [], [], []
def addm(a):
    global mo
    a = np.ascontiguousarray(a, dtype=np.float16); mc.append(a.view(np.uint16).reshape(-1)); rows.append([mo, a.shape[0], a.shape[1]]); mo += a.size
def addv(v):
    global fo
    v = np.ascontiguousarray(v, dtype=np.float32); fc.append(v); rows.append([fo, 1, len(v)]); fo += len(v)
g = lambda key: sd[key].numpy().astype(np.float32)
addm(g('gru.weight_ih_l0').T); addm(g('gru.weight_hh_l0').T); addv(g('gru.bias_ih_l0')); addv(g('gru.bias_hh_l0'))
addm(g('gru.weight_ih_l1').T); addm(g('gru.weight_hh_l1').T); addv(g('gru.bias_ih_l1')); addv(g('gru.bias_hh_l1'))
addm(g('head.weight').T); addv(g('head.bias'))
Wn = dict(W0)
Wn['mats'] = np.concatenate([mats] + mc); Wn['flat'] = np.concatenate([flat] + fc)
Wn['meta'] = np.vstack([meta, np.array(rows, np.int64)]); Wn['mods'] = np.vstack([mods, np.array([[0, r, so, H]], np.int64)])
Wn['coef'] = np.vstack([A * W0['coef'], w[None, :]]).astype(np.float32); Wn['const'] = const_new.astype(np.float32)
ptest = torch_out(net, W0['selftest_x'][None])[0].astype(np.float64)
Wn['selftest_y'] = (A * (W0['selftest_y'].astype(np.float64) - W0['const']) + w * ptest + const_new).astype(np.float32)
os.makedirs(OUT, exist_ok=True); np.savez(f'{OUT}/weights.npz', **Wn); shutil.copy(f'{SRC}/solution.py', f'{OUT}/solution.py')
print('моделей в смеси:', len(Wn['mods']))

# ---------------- проверки ----------------
pm_new = load_pm(OUT, 'sol_new'); print('самопроверка внутри solution.py пройдена')
Xe = np.asarray(XV[20:22], np.float32); worst = 0.0
for i in range(2):                                          # последовательности, которых не было в статистиках
    o_old = run_pm(pm_old, Xe[i], 100 + i).astype(np.float64); o_new = run_pm(pm_new, Xe[i], 100 + i).astype(np.float64)
    exp = A * (o_old - W0['const']) + w * torch_out(net, Xe[i:i + 1])[0] + const_new
    worst = max(worst, np.abs(o_new - exp).max())
print('макс. расхождение с формулой:', worst, '(нужно < 1e-3)')
d1, d2 = run_pm(pm_new, Xe[0][:3000], 300), run_pm(pm_new, Xe[0][:3000], 301)
print('детерминизм (два прогона побитово равны):', np.array_equal(d1, d2), '| все значения конечны:', bool(np.isfinite(d1).all()))
tm = {}
for nm, pm in (('старый', pm_old), ('новый', pm_new)):
    tm[nm] = min((lambda t0: (run_pm(pm, Xe[0], 400 + k), time.time() - t0)[1])(time.time()) for k in range(2))
print(f'на строку: старый {tm["старый"] / L * 1e6:.0f} мкс, новый {tm["новый"] / L * 1e6:.0f} мкс | оценка на платформе ≈ {18.4 * tm["новый"] / tm["старый"]:.0f} мин (грубо, от 18m22s друга; метод раньше завышал примерно на 10%)')

# ---------------- архив ----------------
shutil.make_archive('/content/ens3_submission', 'zip', OUT)
print('архив:', '/content/ens3_submission.zip', f'{os.path.getsize("/content/ens3_submission.zip") / 1e6:.1f} МБ')
try: shutil.copy('/content/ens3_submission.zip', '/content/drive/MyDrive/wnn/')
except Exception as e: print('копия на Drive не удалась:', e)
try:
    from google.colab import files; files.download('/content/ens3_submission.zip')
except Exception as e: print('скачай архив вручную из панели файлов:', e)

наша модель: скрытое состояние 128
старый ансамбль на 20 посл.: 29 с
множитель старой части: [0.675 0.676] | коэффициент нашей модели: [0.248 0.238] | const: [0.233 0.173]
моделей в смеси: 3
самопроверка внутри solution.py пройдена
макс. расхождение с формулой: 0.00014406199087557692 (нужно < 1e-3)
детерминизм (два прогона побитово равны): True | все значения конечны: True
на строку: старый 63 мкс, новый 119 мкс | оценка на платформе ≈ 35 мин (грубо, от 18m22s друга; метод раньше завышал примерно на 10%)
архив: /content/ens3_submission.zip 1.2 МБ


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
%run -i /content/drive/MyDrive/wnn/lib.py
import os, numpy as np, torch
from sklearn.model_selection import StratifiedKFold

Xva = np.load('/content/data/valid300_X.npy', mmap_mode='r'); assert Xva.shape[-1] == 112
Yva = np.load('/content/data/valid300_Y.npy'); Mva = np.load('/content/data/valid300_M.npy').astype(bool)
fam_v = fam_of(Yva, verbose=False); have = np.bincount(fam_v, minlength=4) / len(fam_v); N = len(Yva)

# ---------- предсказания честных моделей (valid300 не видели), с кэшем на Drive ----------
def get_pred(name, ck):
    path = f'{CKPT_DIR}/P_{name}_valid300.npy'
    if os.path.exists(path): return np.load(path)
    if not os.path.exists(f'{CKPT_DIR}/{ck}'): print('нет чекпойнта', ck, '- пропускаю'); return None
    net = Net(128).to(dev); net.load_state_dict(torch.load(f'{CKPT_DIR}/{ck}', map_location=dev)); net.eval()
    with torch.no_grad(): P = predict(net, Xva, np.arange(N)).astype(np.float32)
    np.save(path, P); print('посчитано и сохранено:', name); return P

Ps = {'baseline×0.3': np.load(f'{CKPT_DIR}/Pb_valid300.npy').astype(np.float32) * 0.3}
for name, ck in (('gru6573', 'gru_full_val_best.pt'), ('corr3k', 'corr3k_1_best.pt')):
    P = get_pred(name, ck)
    if P is not None: Ps[name] = P
names = list(Ps); K = len(names); print('модели в стеке:', names)

# ---------- строки метрики: is_scored после прогрева ----------
S = Mva & (np.arange(L) >= WARM)[None]; si, ti = np.where(S); R = len(si)
y = np.clip(Yva[si, ti].astype(np.float64), -2, 2)
fr = fam_v[si]; wmix = (VALMIX / have)[fr]                    # вес строки = |y| · поправка на смесь семейств
F = np.stack([Ps[n][si, ti].astype(np.float64) for n in names], -1)   # (R, 2, K)
one = np.ones((R, 1))
Z = {('own', j): np.hstack([one, F[:, j, :]]) for j in (0, 1)}
Z.update({('cross', j): np.hstack([one, F[:, 0, :], F[:, 1, :]]) for j in (0, 1)})
print('строк в метрике:', R)

def seq_sums(p):                                              # (R,2) -> (N,2,6), суммы по последовательностям
    out = np.zeros((N, 2, 6))
    for j in (0, 1):
        yy, pp = y[:, j], np.clip(p[:, j], -2, 2); a = np.abs(yy) * wmix
        for c, v in enumerate((a, a * yy, a * pp, a * yy * yy, a * pp * pp, a * yy * pp)): out[:, j, c] = np.bincount(si, v, minlength=N)
    return out
def corr_T(T):
    sw = T[..., 0]; my, mp = T[..., 1] / sw, T[..., 2] / sw
    cov = T[..., 5] / sw - my * mp; vy, vp = T[..., 3] / sw - my ** 2, T[..., 4] / sw - mp ** 2
    return (cov / np.sqrt(np.maximum(vy * vp, 1e-16))).mean(-1)
score = lambda p: float(corr_T(seq_sums(p).sum(0)))

def wls(Za, j, rows, lam=1e-6):                               # взвешенный МНК, вес |y|·mix
    w = np.abs(y[rows, j]) * wmix[rows]; A = Za.T @ (Za * w[:, None])
    A += lam * np.trace(A) / len(A) * np.eye(len(A))
    return np.linalg.solve(A, Za.T @ (w * y[rows, j]))

def oof(kind, mode, seed):                                    # mode: 'global' | 'fam' | 'fam_scale'
    P = np.zeros_like(y)
    for tr_s, te_s in StratifiedKFold(5, shuffle=True, random_state=seed).split(np.zeros(N), fam_v):
        tr, te = np.where(np.isin(si, tr_s))[0], np.where(np.isin(si, te_s))[0]
        for j in (0, 1):
            Zj = Z[(kind, j)]
            if mode == 'global': P[te, j] = Zj[te] @ wls(Zj[tr], j, tr)
            elif mode == 'fam':
                for f in range(4):
                    a, b = tr[fr[tr] == f], te[fr[te] == f]; P[b, j] = Zj[b] @ wls(Zj[a], j, a)
            else:
                bt = wls(Zj[tr], j, tr); g_tr, g_te = Zj[tr] @ bt, Zj[te] @ bt
                for f in range(4):
                    a, b = fr[tr] == f, fr[te] == f
                    bf = wls(np.column_stack([np.ones(a.sum()), g_tr[a]]), j, tr[a])
                    P[te[b], j] = np.column_stack([np.ones(b.sum()), g_te[b]]) @ bf
    return P

# ---------- сравнение ----------
print('\nотдельные модели (is_scored, валид. смесь):')
for k, n in enumerate(names): print(f'  {n:14s} {score(F[:, :, k]):.4f}')
VAR = {'среднее моделей': None, 'общий стек': ('own', 'global'), 'стек по семействам': ('own', 'fam'),
       'общий стек + const/масштаб по сем.': ('own', 'fam_scale'),
       'общий стек, перекрёстные': ('cross', 'global'), 'по семействам, перекрёстные': ('cross', 'fam')}
OOF = {}
print('\nвариант | скор по 3 повторам CV | среднее')
for v, spec in VAR.items():
    if spec is None: OOF[v] = [F.mean(-1)]; s = [score(OOF[v][0])]
    else: OOF[v] = [oof(*spec, seed=sd) for sd in range(3)]; s = [score(p) for p in OOF[v]]
    print(f'  {v:36s} {np.round(s, 4)} | {np.mean(s):.4f}')

# ---------- парный бутстрап по последовательностям (внутри семейств) ----------
def boot(Pa, Pb_, B=2000, seed=0):
    Ta, Tb = seq_sums(Pa), seq_sums(Pb_); rng = np.random.default_rng(seed)
    gr = [np.where(fam_v == f)[0] for f in range(4)]; d = np.empty(B)
    for i in range(B):
        ix = np.concatenate([rng.choice(g, len(g)) for g in gr]); d[i] = corr_T(Tb[ix].sum(0)) - corr_T(Ta[ix].sum(0))
    return f'Δ = {d.mean():+.4f}, 90% интервал [{np.percentile(d, 5):+.4f}, {np.percentile(d, 95):+.4f}], P(Δ>0) = {(d > 0).mean():.2f}'
print('\nпротив общего стека:')
for v in ('стек по семействам', 'общий стек + const/масштаб по сем.', 'общий стек, перекрёстные', 'по семействам, перекрёстные'):
    print(f'  {v:36s}', boot(OOF['общий стек'][0], OOF[v][0]))
print(f'  {"общий стек против среднего":36s}', boot(OOF['среднее моделей'][0], OOF['общий стек'][0]))

# ---------- коэффициенты на всех 300 (для понимания, не для сабмита) ----------
print('\nкоэффициенты стека по семействам [const, ' + ', '.join(names) + ']:')
allr = np.arange(R)
for j in (0, 1):
    print(f'  общий  t{j}:', wls(Z[("own", j)], j, allr).round(3))
    for f in range(4):
        a = allr[fr == f]; print(f'  F{f}     t{j}:', wls(Z[("own", j)][a], j, a).round(3))

посчитано и сохранено: gru6573
посчитано и сохранено: corr3k
модели в стеке: ['baseline×0.3', 'gru6573', 'corr3k']
строк в метрике: 565711

отдельные модели (is_scored, валид. смесь):
  baseline×0.3   0.6572
  gru6573        0.6569
  corr3k         0.6541

вариант | скор по 3 повторам CV | среднее
  среднее моделей                      [0.661] | 0.6610
  общий стек                           [0.6589 0.6583 0.6604] | 0.6592
  стек по семействам                   [0.6581 0.6554 0.6609] | 0.6581
  общий стек + const/масштаб по сем.   [0.6586 0.6562 0.6608] | 0.6586
  общий стек, перекрёстные             [0.6581 0.6562 0.6612] | 0.6585
  по семействам, перекрёстные          [0.6548 0.6532 0.6601] | 0.6560

против общего стека:
  стек по семействам                   Δ = -0.0008, 90% интервал [-0.0021, +0.0005], P(Δ>0) = 0.16
  общий стек + const/масштаб по сем.   Δ = -0.0002, 90% интервал [-0.0014, +0.0008], P(Δ>0) = 0.39
  общий стек, перекрёстные             Δ = -0.0007, 90% интервал [-0.0

In [21]:
import os, shutil, subprocess, time
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'): drive.mount('/content/drive')
%run -i /content/drive/MyDrive/wnn/lib.py
D = '/content/data'; os.makedirs(D, exist_ok=True)
sh = lambda c: print(subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip())
sh('df -h /content | tail -1')

# ---------- 1. ищем уже скачанные файлы (в /content и на Drive) ----------
def find(name):
    r = subprocess.run(f"find /content -maxdepth 7 -name '{name}' -not -path '*/.Trash*' 2>/dev/null | head -1",
                       shell=True, capture_output=True, text=True).stdout.strip()
    print(f'{name}: {r or "НЕ НАЙДЕН"}'); return r
tr, va, bo = find('train.parquet'), find('valid.parquet'), find('baseline.onnx')
assert tr and va, 'parquet не найдены ни в /content, ни на Drive'
assert os.path.dirname(tr) == os.path.dirname(va), 'train и valid лежат в разных папках: укажи ROOT вручную'
ROOT = os.path.dirname(tr); print('ROOT =', ROOT)
if bo: shutil.copy(bo, f'{CKPT_DIR}/baseline.onnx')
else: print('baseline.onnx не найден: для дистилляции с baseline положи его в', CKPT_DIR)

# ---------- 2. сборка в float16 ----------
meta = {'seq_ix', 'step_in_seq', 'need_prediction', 'is_scored', 't0', 't1'}
def build16(src, dst, rows):
    pf = pq.ParquetFile(f'{ROOT}/{src}.parquet'); feat = [c for c in pf.schema_arrow.names if c not in meta]; assert len(feat) == NF
    X = np.lib.format.open_memmap(f'{D}/{dst}_X16.npy', 'w+', np.float16, (len(rows), L, NF))
    Y = np.zeros((len(rows), L, 2), np.float32); t = time.time()
    for k, i in enumerate(rows):
        tb = pf.read_row_group(int(i), columns=feat + ['t0', 't1'])
        X[k] = np.column_stack([tb[c].to_numpy() for c in feat]).astype(np.float16)
        Y[k] = np.column_stack([tb['t0'].to_numpy(), tb['t1'].to_numpy()])
        if k % 500 == 0: print(dst, k, '/', len(rows), f'{time.time() - t:.0f} с', flush=True)
    X.flush(); del X; np.save(f'{D}/{dst}_Y.npy', Y)

# train: те же 5000, что раньше (сид 0)
ntr = pq.ParquetFile(f'{ROOT}/train.parquet').num_row_groups; assert ntr == 10607
if not (os.path.exists(f'{D}/trainfull_X16.npy') and os.path.exists(f'{D}/trainfull_Y.npy')):
    build16('train', 'trainfull', np.sort(np.random.default_rng(0).choice(ntr, 5000, replace=False)))

# valid: 300 отложенных (сид 1) и остальные 1573
held = np.sort(np.random.default_rng(1).choice(1873, 300, replace=False))
rest = np.array([i for i in range(1873) if i not in set(held.tolist())])
if not (os.path.exists(f'{D}/validrest_X16.npy') and os.path.exists(f'{D}/validrest_Y.npy')):
    build16('valid', 'validrest', rest)
if not all(os.path.exists(f'{D}/valid300_{s}.npy') for s in 'XYM'):
    build('valid', 'valid300', 300, 1, True)              # функция из lib.py, тот же сид, что раньше

# ---------- 3. проверки и уборка ----------
Yv = np.load(f'{D}/valid300_Y.npy'); print('valid300 семейства:', np.bincount(fam_of(Yv, verbose=False), minlength=4), '(ожидаю около [54 64 84 98])')
print('trainfull:', np.load(f'{D}/trainfull_X16.npy', mmap_mode='r').shape, '| validrest:', np.load(f'{D}/validrest_X16.npy', mmap_mode='r').shape)
os.remove(f'{ROOT}/train.parquet')                        # 28 ГБ больше не нужны; valid.parquet оставляем
sh('df -h /content | tail -1')

overlay         113G   41G   73G  36% /
train.parquet: /content/wnn_connectome_starterpack/datasets/train.parquet
valid.parquet: /content/wnn_connectome_starterpack/datasets/valid.parquet
baseline.onnx: /content/wnn_connectome_starterpack/baseline/baseline.onnx
ROOT = /content/wnn_connectome_starterpack/datasets
trainfull 0 / 5000 0 с
trainfull 500 / 5000 34 с
trainfull 1000 / 5000 68 с
trainfull 1500 / 5000 107 с
trainfull 2000 / 5000 142 с
trainfull 2500 / 5000 177 с
trainfull 3000 / 5000 218 с
trainfull 3500 / 5000 253 с
trainfull 4000 / 5000 288 с
trainfull 4500 / 5000 324 с
validrest 0 / 1573 0 с
validrest 500 / 1573 31 с
validrest 1000 / 1573 70 с
validrest 1500 / 1573 102 с
valid300 семейства: [68 54 80 98] (ожидаю около [54 64 84 98])
trainfull: (5000, 20000, 112) | validrest: (1573, 20000, 112)
overlay         113G   42G   71G  37% /


In [22]:
%run -i /content/drive/MyDrive/wnn/lib.py
!pip install -q onnx
import onnx, subprocess
from onnx import numpy_helper
assert torch.cuda.is_available(), 'нужен GPU: Runtime -> Change runtime type'
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False

USE_BASELINE = True        # False: учитель только из наших моделей (если baseline не разрешат)
ALPHAS = [0.5]             # доля учителя в лоссе; можно [0.5, 1.0], каждый прогон ~20 мин
EPOCHS, SEED = 5, 0

# ---------------- данные ----------------
D = '/content/data'
for p in ('trainfull_X16.npy', 'trainfull_Y.npy', 'validrest_X16.npy', 'valid300_X.npy', 'valid300_Y.npy', 'valid300_M.npy'):
    assert os.path.exists(f'{D}/{p}'), f'нет {D}/{p}: данные из прошлой сессии пропали, их надо пересобрать'
if not os.path.exists(f'{D}/validrest_Y.npy'):            # таргеты 1573 валидационных: раньше они жили только в памяти
    pfv = pq.ParquetFile(f'{ROOT}/valid.parquet'); assert pfv.num_row_groups == 1873
    held = set(np.random.default_rng(1).choice(1873, 300, replace=False).tolist())
    rest = [i for i in range(1873) if i not in held]
    Yr = np.zeros((len(rest), L, 2), np.float32)
    for k, i in enumerate(rest):
        tb = pfv.read_row_group(int(i), columns=['t0', 't1']); Yr[k] = np.column_stack([tb['t0'].to_numpy(), tb['t1'].to_numpy()])
    np.save(f'{D}/validrest_Y.npy', Yr); print('таргеты validrest пересобраны')
XA = np.load(f'{D}/trainfull_X16.npy', mmap_mode='r'); XB = np.load(f'{D}/validrest_X16.npy', mmap_mode='r'); nA = len(XA)
Ytr = np.concatenate([np.load(f'{D}/trainfull_Y.npy'), np.load(f'{D}/validrest_Y.npy')]); N = len(Ytr)
assert N == nA + len(XB)
def getX(idx):                                            # idx отсортирован; float16 -> float32
    idx = np.asarray(idx); a, b = idx[idx < nA], idx[idx >= nA] - nA
    parts = ([XA[a]] if len(a) else []) + ([XB[b]] if len(b) else [])
    return np.concatenate(parts).astype(np.float32)
Xva = np.load(f'{D}/valid300_X.npy', mmap_mode='r'); Yva = np.load(f'{D}/valid300_Y.npy'); Mva = np.load(f'{D}/valid300_M.npy')
fam_v = fam_of(Yva, verbose=False); have = np.bincount(fam_v, minlength=4) / len(fam_v)
print('обучающих последовательностей:', N)

def mixP(P):                                              # скор готовых предсказаний по валидационной смеси
    m = VALMIX / have; Sm = {f: sums_np(P[fam_v == f], Yva[fam_v == f], Mva[fam_v == f]) for f in range(4)}
    return {k: round(corr(sum(m[f] * Sm[f][k] for f in range(4))), 4) for k in ('все строки', 'is_scored')}

# ---------------- модели учителя ----------------
def load_net(ck):
    n = Net(128).to(dev); n.load_state_dict(torch.load(f'{CKPT_DIR}/{ck}', map_location=dev)); n.eval(); return n
teachers = {'gru6573': (load_net('gru_full_val_best.pt'), 1.0), 'corr3k': (load_net('corr3k_1_best.pt'), 1.0)}

if USE_BASELINE:
    found = subprocess.run("find /content -maxdepth 6 -name baseline.onnx 2>/dev/null | head -1", shell=True, capture_output=True, text=True).stdout.strip()
    assert found, 'не нашёл baseline.onnx: положи его в /content/drive/MyDrive/wnn/ (из стартпака, папка baseline)'
    g = onnx.load(found).graph
    init = {i.name: numpy_helper.to_array(i) for i in g.initializer}
    for nd in g.node:
        if nd.op_type == 'Constant':
            for a in nd.attribute:
                if a.name == 'value': init[nd.output[0]] = numpy_helper.to_array(a.t)
    grus = sorted([nd for nd in g.node if nd.op_type == 'GRU'], key=lambda nd: init[nd.input[1]].shape[2])
    assert len(grus) == 2, f'ожидал 2 узла GRU, нашёл {len(grus)}'
    reo = lambda a: np.concatenate([np.split(a, 3, 0)[i] for i in (1, 0, 2)], 0)      # ONNX z,r,h -> PyTorch r,z,n
    sd = {}
    for l, nd in enumerate(grus):
        att = {a.name: onnx.helper.get_attribute_value(a) for a in nd.attribute}
        assert att.get('linear_before_reset', 0) == 1, 'GRU без linear_before_reset: формула не как в PyTorch'
        W, R = init[nd.input[1]][0], init[nd.input[2]][0]; H = R.shape[1]
        B = init[nd.input[3]][0] if len(nd.input) > 3 and nd.input[3] else np.zeros(6 * H, np.float32)
        sd[f'gru.weight_ih_l{l}'], sd[f'gru.weight_hh_l{l}'] = reo(W), reo(R)
        sd[f'gru.bias_ih_l{l}'], sd[f'gru.bias_hh_l{l}'] = reo(B[:3 * H]), reo(B[3 * H:])
    W2 = [v for v in init.values() if v.dtype.kind == 'f' and v.ndim == 2 and sorted(v.shape) == [2, H]]
    b2 = [v for v in init.values() if v.dtype.kind == 'f' and v.ndim == 1 and v.shape == (2,)]
    assert len(W2) == 1 and len(b2) == 1, f'голову не опознал: {len(W2)} матриц, {len(b2)} смещений'
    sd['head.weight'] = W2[0] if W2[0].shape == (2, H) else W2[0].T; sd['head.bias'] = b2[0]
    bnet = Net(H).to(dev); bnet.load_state_dict({k: torch.from_numpy(np.ascontiguousarray(v, np.float32)) for k, v in sd.items()}); bnet.eval()
    Pb = np.load(f'{CKPT_DIR}/Pb_valid300.npy', mmap_mode='r')
    with torch.no_grad(): o, _ = bnet(torch.from_numpy(np.ascontiguousarray(Xva[0:1, :3000])).to(dev))
    diff = np.abs(o[0].cpu().numpy()[WARM:] - Pb[0, WARM:3000]).max()
    print(f'перенос baseline ({found}): макс. расхождение с baseline.onnx {diff:.2e}')
    assert diff < 2e-3, 'перенос не совпал с ONNX: дистилляцию с baseline не запускаю'
    teachers['baseline×0.3'] = (bnet, 0.3)
print('учитель = равное среднее:', list(teachers))

# ---------------- учитель на valid300 (проверка) ----------------
Pv = [np.clip(np.load(f'{CKPT_DIR}/Pb_valid300.npy') * 0.3, -2, 2)] if USE_BASELINE else []
Pv += [np.load(f'{CKPT_DIR}/P_gru6573_valid300.npy'), np.load(f'{CKPT_DIR}/P_corr3k_valid300.npy')]
print('учитель на valid300:', mixP(np.mean(Pv, 0)), '| для сравнения gru6573:', mixP(Pv[-2])); del Pv

# ---------------- учитель на обучающих данных ----------------
tname = f'{D}/teacher_{"b" if USE_BASELINE else "nob"}_T16.npy'
if not os.path.exists(tname):
    T16 = np.lib.format.open_memmap(tname + '.tmp.npy', 'w+', np.float16, (N, L, 2)); t0 = time.time(); bs = 32
    with torch.no_grad():
        for s in range(0, N, bs):
            idx = np.arange(s, min(s + bs, N)); x = torch.from_numpy(getX(idx)).to(dev); acc = 0
            for net, sc in teachers.values():
                h, o_ = None, []
                for t in range(0, L, 2000): o, h = net(x[:, t:t + 2000], h); o_.append(o)
                acc = acc + (torch.cat(o_, 1) * sc).clamp(-2, 2)
            T16[idx] = (acc / len(teachers)).cpu().numpy().astype(np.float16)
            if s % (bs * 40) == 0: print(f'учитель {s}/{N} {time.time() - t0:.0f} с', flush=True)
    T16.flush(); del T16; os.rename(tname + '.tmp.npy', tname)
T16 = np.load(tname, mmap_mode='r'); print('учитель готов:', tname)

# ---------------- обучение ученика ----------------
def train_distill(alpha, seed, tag, epochs=EPOCHS, bs=32, ch=1000, lr=2e-3):
    torch.manual_seed(seed); model = Net(128).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * (N // bs) * (L // ch), pct_start=0.05)
    rng = np.random.default_rng(seed); best = -1
    for ep in range(epochs):
        t0 = time.time(); order = rng.permutation(N); run = []; model.train()
        for gi in range(N // bs):
            idx = np.sort(order[gi * bs:(gi + 1) * bs])
            x = torch.from_numpy(getX(idx)).to(dev); y = torch.from_numpy(Ytr[idx]).to(dev).clamp(-2, 2)
            tt = torch.from_numpy(np.asarray(T16[idx], np.float32)).to(dev); h = None
            for t in range(0, L, ch):
                o, h = model(x[:, t:t + ch], h); h = h.detach()
                yy, ty = y[:, t:t + ch], tt[:, t:t + ch]
                l = yy.abs() * ((1 - alpha) * (o - yy) ** 2 + alpha * (o - ty) ** 2)
                if t < WARM: l = l[:, WARM - t:]
                loss = l.mean(); opt.zero_grad(set_to_none=True); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); run.append(loss.item())
        r = mix_score(model)
        print(f'α={alpha} эпоха {ep + 1}/{epochs} | {time.time() - t0:.0f} с | loss {np.mean(run):.4f} | valid300 {r}', flush=True)
        torch.save(model.state_dict(), f'{CKPT_DIR}/{tag}_last.pt')
        if r['is_scored'] > best: best = r['is_scored']; torch.save(model.state_dict(), f'{CKPT_DIR}/{tag}_best.pt')
    return model, best

for a in ALPHAS:
    tag = f'distill_{"b" if USE_BASELINE else "nob"}_a{a}_s{SEED}'
    _, best = train_distill(a, SEED, tag)
    print(f'итог {tag}: лучшая эпоха {best:.4f} | ориентиры: учитель 0.6610, gru6573 0.6569, baseline×0.3 0.6572')

обучающих последовательностей: 6573
перенос baseline (/content/drive/MyDrive/wnn/baseline.onnx): макс. расхождение с baseline.onnx 2.22e-05
учитель = равное среднее: ['gru6573', 'corr3k', 'baseline×0.3']
учитель на valid300: {'все строки': 0.5041, 'is_scored': 0.661} | для сравнения gru6573: {'все строки': 0.4982, 'is_scored': 0.6569}
учитель 0/6573 1 с
учитель 1280/6573 49 с
учитель 2560/6573 96 с
учитель 3840/6573 144 с
учитель 5120/6573 191 с
учитель 6400/6573 239 с
учитель готов: /content/data/teacher_b_T16.npy
α=0.5 эпоха 1/5 | 275 с | loss 0.5007 | valid300 {'все строки': 0.489, 'is_scored': 0.651}
α=0.5 эпоха 2/5 | 268 с | loss 0.4733 | valid300 {'все строки': 0.497, 'is_scored': 0.657}
α=0.5 эпоха 3/5 | 271 с | loss 0.4663 | valid300 {'все строки': 0.497, 'is_scored': 0.652}
α=0.5 эпоха 4/5 | 274 с | loss 0.4591 | valid300 {'все строки': 0.503, 'is_scored': 0.658}
α=0.5 эпоха 5/5 | 273 с | loss 0.4521 | valid300 {'все строки': 0.504, 'is_scored': 0.658}
итог distill_b_a0.5_s0: 

In [23]:
st = Net(128).to(dev); st.load_state_dict(torch.load(f'{CKPT_DIR}/distill_b_a0.5_s0_last.pt', map_location=dev)); st.eval()
p_path = f'{CKPT_DIR}/P_distill05_valid300.npy'
if not os.path.exists(p_path): np.save(p_path, predict(st, Xva, np.arange(len(Xva))).astype(np.float32))
P = {'base': np.clip(np.load(f'{CKPT_DIR}/Pb_valid300.npy') * 0.3, -2, 2), 'gru': np.load(f'{CKPT_DIR}/P_gru6573_valid300.npy'),
     'corr': np.load(f'{CKPT_DIR}/P_corr3k_valid300.npy'), 'stud': np.load(p_path)}
for combo in (['stud'], ['gru'], ['gru', 'stud'], ['base', 'gru', 'corr'], ['base', 'gru', 'corr', 'stud'], ['base', 'corr', 'stud']):
    print(f'{" + ".join(combo):24s}', mixP(np.mean([P[k] for k in combo], 0)))

stud                     {'все строки': 0.5036, 'is_scored': 0.6584}
gru                      {'все строки': 0.4982, 'is_scored': 0.6569}
gru + stud               {'все строки': 0.5038, 'is_scored': 0.6593}
base + gru + corr        {'все строки': 0.5041, 'is_scored': 0.661}
base + gru + corr + stud {'все строки': 0.5054, 'is_scored': 0.6612}
base + corr + stud       {'все строки': 0.5046, 'is_scored': 0.6609}


In [24]:
for v in ('getX', 'Ytr', 'N', 'mixP', 'P'):
    assert v in globals(), f'нет {v}: прогони блок данных из ячейки дистилляции и ячейку проверки ученика'

POWER = 1.0                     # степень поправки: 1.0 = точно под смесь valid, 0.5 = на полпути
fam_tr = fam_of(Ytr, verbose=False); trmix = np.bincount(fam_tr, minlength=4) / N
wf = (VALMIX / trmix) ** POWER; wseq = wf[fam_tr]; wseq = wseq / wseq.mean()
print('доли train:', trmix.round(3), '| valid:', VALMIX.round(3), '| вес семейства:', (wf / (wf[fam_tr]).mean()).round(2))

def train_rw(seed, tag, epochs=5, bs=32, ch=1000, lr=2e-3):
    torch.manual_seed(seed); model = Net(128).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * (N // bs) * (L // ch), pct_start=0.05)
    rng = np.random.default_rng(seed); W = torch.from_numpy(wseq.astype(np.float32)).to(dev)
    for ep in range(epochs):
        t0 = time.time(); order = rng.permutation(N); run = []; model.train()
        for gi in range(N // bs):
            idx = np.sort(order[gi * bs:(gi + 1) * bs])
            x = torch.from_numpy(getX(idx)).to(dev); y = torch.from_numpy(Ytr[idx]).to(dev).clamp(-2, 2)
            ws = W[torch.from_numpy(idx).to(dev)][:, None, None]; h = None
            for t in range(0, L, ch):
                o, h = model(x[:, t:t + ch], h); h = h.detach(); yy = y[:, t:t + ch]
                l = ws * yy.abs() * (o - yy) ** 2
                if t < WARM: l = l[:, WARM - t:]
                loss = l.mean(); opt.zero_grad(set_to_none=True); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); run.append(loss.item())
        r = mix_score(model)
        print(f'эпоха {ep + 1}/{epochs} | {time.time() - t0:.0f} с | loss {np.mean(run):.4f} | valid300 {r}', flush=True)
        torch.save(model.state_dict(), f'{CKPT_DIR}/{tag}_last.pt')
    return model

tag = f'gru_rw_p{POWER}_s0'
net_rw = train_rw(0, tag)

# ---------- сравнение: одиночно, по семействам и в ансамбле ----------
P['rw'] = predict(net_rw, Xva, np.arange(len(Xva))).astype(np.float32); np.save(f'{CKPT_DIR}/P_{tag}_valid300.npy', P['rw'])
print('\nпо семействам (is_scored):')
for k in ('gru', 'rw'):
    print(f'  {k:4s}', [round(corr(sums_np(P[k][fam_v == f], Yva[fam_v == f], Mva[fam_v == f])['is_scored']), 3) for f in range(4)])
print('\nкомбинации (валидационная смесь):')
for combo in (['gru'], ['rw'], ['gru', 'rw'], ['base', 'gru', 'corr'], ['base', 'gru', 'corr', 'rw'], ['base', 'corr', 'rw']):
    print(f'  {" + ".join(combo):24s}', mixP(np.mean([P[k] for k in combo], 0)))

доли train: [0.143 0.135 0.296 0.426] | valid: [0.212 0.185 0.23  0.372] | вес семейства: [1.48 1.37 0.78 0.87]
эпоха 1/5 | 318 с | loss 0.9133 | valid300 {'все строки': 0.475, 'is_scored': 0.641}
эпоха 2/5 | 312 с | loss 0.8760 | valid300 {'все строки': 0.488, 'is_scored': 0.65}
эпоха 3/5 | 318 с | loss 0.8630 | valid300 {'все строки': 0.488, 'is_scored': 0.646}
эпоха 4/5 | 316 с | loss 0.8498 | valid300 {'все строки': 0.499, 'is_scored': 0.658}
эпоха 5/5 | 316 с | loss 0.8357 | valid300 {'все строки': 0.499, 'is_scored': 0.658}

по семействам (is_scored):
  gru  [0.78, 0.753, 0.716, 0.393]
  rw   [0.783, 0.755, 0.712, 0.392]

комбинации (валидационная смесь):
  gru                      {'все строки': 0.4982, 'is_scored': 0.6569}
  rw                       {'все строки': 0.4986, 'is_scored': 0.6579}
  gru + rw                 {'все строки': 0.502, 'is_scored': 0.6595}
  base + gru + corr        {'все строки': 0.5041, 'is_scored': 0.661}
  base + gru + corr + rw   {'все строки': 0.505,

In [25]:
import numpy as np, time
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import StratifiedKFold

Xv = np.load('/content/data/valid300_X.npy', mmap_mode='r')
Yv = np.load('/content/data/valid300_Y.npy'); Mv = np.load('/content/data/valid300_M.npy').astype(bool)
famv = fam_of(Yv, verbose=False); hv = np.bincount(famv, minlength=4) / len(famv); NS = len(Yv)
S = Mv & (np.arange(L) >= WARM)[None]; qs, qt = np.where(S); R = len(qs)
yq = np.clip(Yv[qs, qt].astype(np.float64), -2, 2); wq = (VALMIX / hv)[famv[qs]]

# ---------- предсказания ансамбля (учитель: равное среднее трёх честных моделей) ----------
PT = np.mean([np.clip(np.load(f'{CKPT_DIR}/Pb_valid300.npy') * 0.3, -2, 2), np.load(f'{CKPT_DIR}/P_gru6573_valid300.npy'),
              np.load(f'{CKPT_DIR}/P_corr3k_valid300.npy')], 0)[qs, qt].astype(np.float32)

# ---------- счётчики (причинные: на шаге t используются строки 0..t) ----------
PR, AC, STUB = list(range(0, 22)), list(range(104, 112)), -5.199337
d0 = np.concatenate([np.abs(np.diff(np.asarray(Xv[s][:, PR]), axis=0)) for s in range(30)])
QB = np.array([np.quantile(c[c > 0], 0.95) for c in d0.T], np.float32)      # порог «крупного скачка» по колонке
tt = np.arange(L)[:, None]
def since(E):
    last = np.maximum.accumulate(np.where(E, tt, -1), axis=0)
    return np.where(last >= 0, tt - last, tt + 1)
def counters(x):
    d = np.diff(x, axis=0, prepend=x[:1]); st = np.isclose(x[:, 105:108], STUB, atol=1e-3)
    stch = np.vstack([np.zeros((1, 3), bool), st[1:] != st[:-1]])
    E = np.concatenate([d[:, PR] > 0, d[:, PR] < 0, np.abs(d[:, PR]) > QB, d[:, AC] != 0, stch, d[:, [44]] != 0], 1)
    return np.log1p(since(E)).astype(np.float32)
t0 = time.time(); C = np.zeros((R, 78), np.float32)
for s in range(NS):
    r = np.where(qs == s)[0]
    if len(r): C[r] = counters(np.asarray(Xv[s], np.float32))[qt[r]]
print(f'счётчики: {C.shape}, {time.time() - t0:.0f} с')

base = np.column_stack([PT, famv[qs].astype(np.float32)])
VARS = {'контроль (ансамбль + семейство)': base,
        '+ номер шага': np.column_stack([base, np.log1p(qt).astype(np.float32)]),
        '+ шаг + счётчики': np.column_stack([base, np.log1p(qt).astype(np.float32), C])}

# ---------- метрика ----------
def seq_sums(p):
    out = np.zeros((NS, 2, 6))
    for j in (0, 1):
        yy, pp = yq[:, j], np.clip(p[:, j], -2, 2); a = np.abs(yy) * wq
        for c, v in enumerate((a, a * yy, a * pp, a * yy * yy, a * pp * pp, a * yy * pp)): out[:, j, c] = np.bincount(qs, v, minlength=NS)
    return out
def corrT(T):
    sw = T[..., 0]; my, mp = T[..., 1] / sw, T[..., 2] / sw
    return ((T[..., 5] / sw - my * mp) / np.sqrt(np.maximum((T[..., 3] / sw - my ** 2) * (T[..., 4] / sw - mp ** 2), 1e-16))).mean(-1)
sc_all = lambda p: float(corrT(seq_sums(p).sum(0)))
sc_f3 = lambda p: float(corrT(seq_sums(p)[famv == 3].sum(0)))

# ---------- бустинг с CV по последовательностям ----------
rng = np.random.default_rng(0); OOF = {}
for name, Fm in VARS.items():
    P = np.zeros((R, 2)); t0 = time.time()
    for tr_s, te_s in StratifiedKFold(5, shuffle=True, random_state=0).split(np.zeros(NS), famv):
        tr, te = np.where(np.isin(qs, tr_s))[0], np.where(np.isin(qs, te_s))[0]
        sub = rng.choice(tr, min(250_000, len(tr)), replace=False)
        for j in (0, 1):
            m = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=200, random_state=0)
            m.fit(Fm[sub], yq[sub, j], sample_weight=np.abs(yq[sub, j]) * wq[sub]); P[te, j] = m.predict(Fm[te])
    OOF[name] = P; print(f'{name:34s} все сем. {sc_all(P):.4f} | F3 {sc_f3(P):.4f} | {time.time() - t0:.0f} с', flush=True)
print(f'{"ансамбль как есть (без бустинга)":34s} все сем. {sc_all(PT):.4f} | F3 {sc_f3(PT):.4f}')

def boot(Pa, Pb_, B=2000):
    Ta, Tb = seq_sums(Pa), seq_sums(Pb_); r = np.random.default_rng(1); gr = [np.where(famv == f)[0] for f in range(4)]; d = np.empty(B)
    for i in range(B):
        ix = np.concatenate([r.choice(g, len(g)) for g in gr]); d[i] = corrT(Tb[ix].sum(0)) - corrT(Ta[ix].sum(0))
    return f'Δ = {d.mean():+.4f}, 90% [{np.percentile(d, 5):+.4f}, {np.percentile(d, 95):+.4f}], P(Δ>0) = {(d > 0).mean():.2f}'
k = list(VARS)
print('\nсчётчики против шага:  ', boot(OOF[k[1]], OOF[k[2]]))
print('шаг против контроля:   ', boot(OOF[k[0]], OOF[k[1]]))

счётчики: (565711, 78), 19 с
контроль (ансамбль + семейство)    все сем. 0.6510 | F3 0.3658 | 45 с
+ номер шага                       все сем. 0.6279 | F3 0.3134 | 56 с
+ шаг + счётчики                   все сем. 0.6317 | F3 0.3240 | 278 с
ансамбль как есть (без бустинга)   все сем. 0.6610 | F3 0.4022

счётчики против шага:   Δ = +0.0039, 90% [-0.0101, +0.0156], P(Δ>0) = 0.71
шаг против контроля:    Δ = -0.0233, 90% [-0.0388, -0.0080], P(Δ>0) = 0.01


In [26]:
from sklearn.model_selection import StratifiedKFold
Cc = np.minimum(C, np.log1p(2000)).astype(np.float32)          # «давно не было» = 2000 шагов, без утечки номера шага
folds = list(StratifiedKFold(5, shuffle=True, random_state=0).split(np.zeros(NS), famv))
rows = [(np.where(np.isin(qs, a))[0], np.where(np.isin(qs, b))[0]) for a, b in folds]

# ---------- 1. линейный стек с ridge ----------
def wls(Z, yv, w, lam):
    A = Z.T @ (Z * w[:, None]); pen = np.ones(len(A)); pen[0] = 0
    A[np.diag_indices_from(A)] += lam * pen * np.trace(A) / len(A)
    return np.linalg.solve(A, Z.T @ (w * yv))
def lin_oof(with_c, lam):
    P = np.zeros((R, 2))
    for tr, te in rows:
        if with_c:
            mu, sd = Cc[tr].mean(0), Cc[tr].std(0) + 1e-6; Cs = (Cc - mu) / sd
            Z = np.column_stack([np.ones(R), PT, Cs])
        else: Z = np.column_stack([np.ones(R), PT])
        for j in (0, 1):
            w = np.abs(yq[tr, j]) * wq[tr]; P[te, j] = Z[te] @ wls(Z[tr], yq[tr, j], w, lam)
    return P
print('линейный стек:')
L0 = lin_oof(False, 1e-6); print(f'  контроль (ансамбль)          все {sc_all(L0):.4f} | F3 {sc_f3(L0):.4f}')
best = None
for lam in (1e-3, 1e-2, 1e-1, 1.0):
    P = lin_oof(True, lam); s = sc_all(P); print(f'  + счётчики, λ={lam:<6}          все {s:.4f} | F3 {sc_f3(P):.4f}')
    if best is None or s > best[0]: best = (s, lam, P)
print(f'  лучший λ={best[1]} против контроля:', boot(L0, best[2]), '(λ выбран по тем же данным, слегка завышено)')

# ---------- 2. бустинг с сильной регуляризацией ----------
def gb_oof(Fm):
    P = np.zeros((R, 2)); rg = np.random.default_rng(0)
    for tr, te in rows:
        sub = rg.choice(tr, min(250_000, len(tr)), replace=False)
        for j in (0, 1):
            m = HistGradientBoostingRegressor(max_iter=150, learning_rate=0.03, max_leaf_nodes=15, min_samples_leaf=5000,
                                              l2_regularization=1.0, random_state=0)
            m.fit(Fm[sub], yq[sub, j], sample_weight=np.abs(yq[sub, j]) * wq[sub]); P[te, j] = m.predict(Fm[te])
    return P
print('\nбустинг (≥5000 строк в листе):')
G0 = gb_oof(base); print(f'  контроль (ансамбль + сем.)    все {sc_all(G0):.4f} | F3 {sc_f3(G0):.4f}')
G1 = gb_oof(np.column_stack([base, Cc])); print(f'  + счётчики                    все {sc_all(G1):.4f} | F3 {sc_f3(G1):.4f}')
print('  счётчики против контроля:', boot(G0, G1))
print(f'\nориентир: ансамбль как есть   все {sc_all(PT):.4f} | F3 {sc_f3(PT):.4f}')

линейный стек:
  контроль (ансамбль)          все 0.6589 | F3 0.3910
  + счётчики, λ=0.001           все 0.6549 | F3 0.3797
  + счётчики, λ=0.01            все 0.6549 | F3 0.3795
  + счётчики, λ=0.1             все 0.6510 | F3 0.3747
  + счётчики, λ=1.0             все 0.6199 | F3 0.3319
  лучший λ=0.001 против контроля: Δ = -0.0039, 90% [-0.0067, -0.0012], P(Δ>0) = 0.00 (λ выбран по тем же данным, слегка завышено)

бустинг (≥5000 строк в листе):
  контроль (ансамбль + сем.)    все 0.6547 | F3 0.3764
  + счётчики                    все 0.6493 | F3 0.3695
  счётчики против контроля: Δ = -0.0054, 90% [-0.0195, +0.0074], P(Δ>0) = 0.26

ориентир: ансамбль как есть   все 0.6610 | F3 0.4022


In [27]:
for v in ('qs', 'qt', 'famv', 'sc_all', 'sc_f3', 'boot'):
    assert v in globals(), f'нет {v}: прогони начало ячейки со счётчиками (до функции boot)'

PE = np.mean([np.clip(np.load(f'{CKPT_DIR}/Pb_valid300.npy') * 0.3, -2, 2), np.load(f'{CKPT_DIR}/P_gru6573_valid300.npy'),
              np.load(f'{CKPT_DIR}/P_corr3k_valid300.npy')], 0).astype(np.float64)      # (300, L, 2), ансамбль
rows = lambda A: A[qs, qt]
base_r = rows(PE); s0 = sc_all(base_r)
print(f'ансамбль как есть: все {s0:.4f} | F3 {sc_f3(base_r):.4f}')
print('доля строк, где оба прогноза > 0:', round(float(((base_r[:, 0] > 0) & (base_r[:, 1] > 0)).mean()), 3))

def run(title, grid, fn):
    print(f'\n{title}'); best = (s0, None, base_r)
    for g in grid:
        p = fn(g); s = sc_all(p); print(f'  {g:>6}: все {s:.4f} ({s - s0:+.4f}) | F3 {sc_f3(p):.4f}')
        if s > best[0]: best = (s, g, p)
    if best[1] is not None: print(f'  лучший {best[1]} против ансамбля:', boot(base_r, best[2]))
    return best

# 1) степенное преобразование
run('степень γ: sign(p)·|p|^γ', (0.5, 0.7, 0.85, 1.15, 1.3, 1.5, 2.0),
    lambda g: np.sign(base_r) * np.abs(base_r) ** g)

# 2) взаимное усиление таргетов
run('перекрёстно α: p0 = (1−α)p0 − α·p1', (0.05, 0.1, 0.2, 0.3, 0.5),
    lambda a: np.column_stack([(1 - a) * base_r[:, 0] - a * base_r[:, 1], (1 - a) * base_r[:, 1] - a * base_r[:, 0]]))

# 3) причинное вычитание среднего по последовательности
cm = np.cumsum(PE, axis=1) / np.arange(1, L + 1)[None, :, None]; cm_r = rows(cm)
run('вычитание среднего λ: p − λ·mean(p[0..t])', (-0.5, -0.25, 0.25, 0.5, 0.75, 1.0),
    lambda lam: base_r - lam * cm_r)

# 4) проекция на допустимую область (оба > 0 невозможно)
mn = np.maximum(0, base_r.min(1, keepdims=True))
run('проекция κ: p −= κ·max(0, min(p0, p1))', (0.5, 1.0, 1.5), lambda k: base_r - k * mn)

ансамбль как есть: все 0.6610 | F3 0.4022
доля строк, где оба прогноза > 0: 0.075

степень γ: sign(p)·|p|^γ
     0.5: все 0.6512 (-0.0098) | F3 0.3852
     0.7: все 0.6586 (-0.0024) | F3 0.3970
    0.85: все 0.6608 (-0.0002) | F3 0.4010
    1.15: все 0.6598 (-0.0012) | F3 0.4013
     1.3: все 0.6577 (-0.0034) | F3 0.3989
     1.5: все 0.6543 (-0.0068) | F3 0.3941
     2.0: все 0.6451 (-0.0159) | F3 0.3787

перекрёстно α: p0 = (1−α)p0 − α·p1
    0.05: все 0.6620 (+0.0010) | F3 0.4024
     0.1: все 0.6619 (+0.0009) | F3 0.4024
     0.2: все 0.6578 (-0.0032) | F3 0.4021
     0.3: все 0.6473 (-0.0137) | F3 0.4013
     0.5: все 0.6036 (-0.0574) | F3 0.3981
  лучший 0.05 против ансамбля: Δ = +0.0010, 90% [+0.0006, +0.0014], P(Δ>0) = 1.00

вычитание среднего λ: p − λ·mean(p[0..t])
    -0.5: все 0.6488 (-0.0123) | F3 0.4057
   -0.25: все 0.6566 (-0.0045) | F3 0.4042
    0.25: все 0.6606 (-0.0004) | F3 0.3997
     0.5: все 0.6535 (-0.0076) | F3 0.3966
    0.75: все 0.6379 (-0.0231) | F3 0.3931


(0.6612794985616632,
 0.5,
 array([[ 0.31307721, -0.27228275],
        [ 0.67379761, -0.54005945],
        [ 0.19317029, -0.48285854],
        ...,
        [-0.60946161, -0.13277319],
        [ 0.0429513 , -0.66971773],
        [-0.00088946, -0.69601172]], shape=(565711, 2)))

In [29]:
import numpy as np, os, shutil, zipfile, importlib.util, time
from types import SimpleNamespace
A, OUT = 0.07, '/content/ens3x'
SRC = '/content/ens3'
if not os.path.exists(f'{SRC}/weights.npz'):
    SRC = '/content/ens3_from_drive'; os.makedirs(SRC, exist_ok=True)
    zipfile.ZipFile('/content/drive/MyDrive/wnn/ens3_submission.zip').extractall(SRC)
assert os.path.exists(f'{SRC}/weights.npz') and os.path.exists(f'{SRC}/solution.py')

# веса без изменений; смешивание t0/t1 дописываем в конец solution.py поверх класса
if os.path.exists(OUT): shutil.rmtree(OUT)
os.makedirs(OUT); shutil.copy(f'{SRC}/weights.npz', f'{OUT}/weights.npz')
wrap = f'''

# ---- перекрёстное смешивание таргетов: t0' = (1-A)t0 - A*t1, t1' = (1-A)t1 - A*t0, A = {A} ----
_BasePredictionModel = PredictionModel
class PredictionModel(_BasePredictionModel):
    _C, _A = {1 - A!r}, {A!r}
    def predict(self, data_point):
        out = super().predict(data_point)
        if out is None: return None
        o0, o1 = float(out[0]), float(out[1])
        return np.array((self._C * o0 - self._A * o1, self._C * o1 - self._A * o0), dtype=np.float32)
'''
src = open(f'{SRC}/solution.py').read()
assert 'import numpy as np' in src, 'в solution.py нет "import numpy as np": обёртку надо поправить'
open(f'{OUT}/solution.py', 'w').write(src + wrap)

def load_pm(d, name):
    sp = importlib.util.spec_from_file_location(name, f'{d}/solution.py'); m = importlib.util.module_from_spec(sp); sp.loader.exec_module(m)
    return m.PredictionModel()
def run_pm(pm, X, sid):
    return np.stack([pm.predict(SimpleNamespace(seq_ix=sid, state=X[t], need_prediction=True)) for t in range(len(X))]).astype(np.float64)
pm_old, pm_new = load_pm(SRC, 'sol_a'), load_pm(OUT, 'sol_b'); print('самопроверка solution.py пройдена')

M = np.array([[1 - A, -A], [-A, 1 - A]])
X = np.asarray(np.load('/content/data/valid300_X.npy', mmap_mode='r')[0, :3000], np.float32)
o_old, o_new = run_pm(pm_old, X, 1), run_pm(pm_new, X, 1)
print('макс. расхождение с формулой:', np.abs(o_new - o_old @ M).max(), '(нужно < 1e-5) | конечны:', bool(np.isfinite(o_new).all()))
d1, d2 = run_pm(pm_new, X[:1000], 2), run_pm(pm_new, X[:1000], 3); print('детерминизм:', np.array_equal(d1, d2))
t = time.time(); run_pm(pm_old, X, 4); t_old = time.time() - t
t = time.time(); run_pm(pm_new, X, 5); t_new = time.time() - t
print(f'на строку: было {t_old / len(X) * 1e6:.1f} мкс, стало {t_new / len(X) * 1e6:.1f} мкс')

shutil.make_archive('/content/ens3x_submission', 'zip', OUT)
shutil.copy('/content/ens3x_submission.zip', '/content/drive/MyDrive/wnn/'); print('готово: /content/ens3x_submission.zip (копия на Drive)')

самопроверка solution.py пройдена
макс. расхождение с формулой: 5.275011072392033e-08 (нужно < 1e-5) | конечны: True
детерминизм: True
на строку: было 149.1 мкс, стало 151.7 мкс
готово: /content/ens3x_submission.zip (копия на Drive)


In [30]:
def wcorr(y, p, w):
    sw = w.sum(); my, mp = (w * y).sum() / sw, (w * p).sum() / sw
    return (w * (y - my) * (p - mp)).sum() / np.sqrt((w * (y - my) ** 2).sum() * (w * (p - mp) ** 2).sum())
for lo, hi in ((99, 2000), (2000, 5000), (5000, 10000), (10000, 15000), (15000, 20000)):
    m = (qt >= lo) & (qt < hi)
    s = np.mean([wcorr(yq[m, j], base_r[m, j], np.abs(yq[m, j]) * wq[m]) for j in (0, 1)])
    print(f'шаги {lo:>5}–{hi:<5} | строк {m.sum():>6} | скор ансамбля {s:.4f}')

шаги    99–2000  | строк  50668 | скор ансамбля 0.6445
шаги  2000–5000  | строк  79641 | скор ансамбля 0.6854
шаги  5000–10000 | строк 144279 | скор ансамбля 0.6715
шаги 10000–15000 | строк 144369 | скор ансамбля 0.6587
шаги 15000–20000 | строк 146754 | скор ансамбля 0.6560


In [31]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, precision_recall_curve

XT = np.load('/content/data/trainfull_X16.npy', mmap_mode='r'); YT = np.load('/content/data/trainfull_Y.npy')
Xv = np.load('/content/data/valid300_X.npy', mmap_mode='r'); Yv = np.load('/content/data/valid300_Y.npy')
f3_tr = np.where(fam_of(YT, verbose=False) == 3)[0][:800]; f3_te = np.where(fam_of(Yv, verbose=False) == 3)[0]
print('F3: обучение', len(f3_tr), '| проверка', len(f3_te))

GR = [range(0, 22), range(22, 44), range(44, 52), range(52, 74), range(74, 96), range(96, 104), range(104, 112)]
DS, PRE, THR = (0, 2, 10, 20, 50), 50, -0.12
CI = list(range(0, 22)) + list(range(104, 112))
def starts(y):
    e = code3(np.clip(y, -2, 2)) <= -3; return e & ~np.r_[False, e[:-1]]

def seq_feats(x, y, neg_frac, rng):
    d = np.diff(x, axis=0, prepend=x[:1]); on = starts(y[:, 0]) | starts(y[:, 1])
    ta = np.arange(WARM + PRE, L - max(DS) - 2); cand = ta[d[ta, 10] < THR]; lab = on[cand]
    stats = (int(on[ta].sum()), int(lab.sum()))                        # событий в диапазоне, из них среди кандидатов
    if neg_frac < 1:
        keep = lab | (rng.random(len(cand)) < neg_frac); cand, lab = cand[keep], lab[keep]
    G = np.stack([np.abs(d[:, list(g)]).mean(1) for g in GR], 1); CG = np.vstack([np.zeros((1, 7)), np.cumsum(G, 0)])
    core = np.concatenate([d[cand], x[cand][:, 104:112], (CG[cand] - CG[cand - PRE]) / PRE], 1)
    out = {0: core}
    for D_ in DS[1:]:
        post = (CG[cand + D_ + 1] - CG[cand + 1]) / D_
        dx = x[cand + D_][:, CI] - x[cand][:, CI]
        near = np.concatenate([d[cand + k][:, :22] for k in (1, 2)], 1)
        out[D_] = np.concatenate([core, post, dx, near], 1)
    return out, lab, stats

def collect(X, Y, ids, neg_frac):
    rng = np.random.default_rng(0); F = {D_: [] for D_ in DS}; Lb = []; ev = [0, 0]; t = time.time()
    for k, s in enumerate(ids):
        o, lab, st = seq_feats(np.asarray(X[s], np.float32), Y[s], neg_frac, rng)
        for D_ in DS: F[D_].append(o[D_].astype(np.float32))
        Lb.append(lab); ev[0] += st[0]; ev[1] += st[1]
        if k % 200 == 0: print(f'  {k}/{len(ids)} {time.time() - t:.0f} с', flush=True)
    return {D_: np.concatenate(F[D_]) for D_ in DS}, np.concatenate(Lb), ev

Ftr, ltr, evtr = collect(XT, YT, f3_tr, 0.1)                           # в обучении 10% отрицательных
Fte, lte, evte = collect(Xv, Yv, f3_te, 1.0)                           # в проверке все кандидаты
print(f'фильтр кандидатов ловит {evte[1] / evte[0]:.3f} событий | кандидатов в проверке {len(lte)}, из них событий {lte.mean():.4f}')

for D_ in DS:
    clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, max_leaf_nodes=63, random_state=0).fit(Ftr[D_], ltr)
    p = clf.predict_proba(Fte[D_])[:, 1]; pr, rc, _ = precision_recall_curve(lte, p)
    at = lambda r: pr[rc >= r].max()
    print(f'задержка D={D_:>2} | признаков {Ftr[D_].shape[1]:>3} | AP {average_precision_score(lte, p):.3f} | '
          f'точность при полноте 0.3 / 0.5 / 0.7: {at(0.3):.3f} / {at(0.5):.3f} / {at(0.7):.3f}', flush=True)

F3: обучение 800 | проверка 98
  0/800 0 с
  200/800 7 с
  400/800 15 с
  600/800 23 с
  0/98 0 с
фильтр кандидатов ловит 0.957 событий | кандидатов в проверке 206873, из них событий 0.0216
задержка D= 0 | признаков 127 | AP 0.129 | точность при полноте 0.3 / 0.5 / 0.7: 0.153 / 0.140 / 0.120
задержка D= 2 | признаков 208 | AP 0.137 | точность при полноте 0.3 / 0.5 / 0.7: 0.159 / 0.144 / 0.128
задержка D=10 | признаков 208 | AP 0.175 | точность при полноте 0.3 / 0.5 / 0.7: 0.186 / 0.170 / 0.161
задержка D=20 | признаков 208 | AP 0.244 | точность при полноте 0.3 / 0.5 / 0.7: 0.273 / 0.218 / 0.188
задержка D=50 | признаков 208 | AP 0.300 | точность при полноте 0.3 / 0.5 / 0.7: 0.364 / 0.270 / 0.222


In [32]:
from sklearn.model_selection import KFold
DUSE = (2, 10, 20, 50)
t0 = time.time()
clfs = {D_: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, max_leaf_nodes=63, random_state=0).fit(Ftr[D_], ltr) for D_ in DUSE}
print(f'детекторы обучены: {time.time() - t0:.0f} с')
calib = lambda p, frac=0.1: (lambda o: o / (1 + o))(p / np.maximum(1 - p, 1e-9) * frac)   # поправка на 10% отрицательных
LB = [(2, 10), (10, 20), (20, 50), (50, 100), (100, 200), (200, 500), (500, 1000), (1000, 2000)]
DFOR = lambda a: max(D_ for D_ in DUSE if D_ <= a)          # самый «зрячий» детектор, доступный на этом лаге
tt = np.arange(L)

feat3 = {}
for s in f3_te:
    x = np.asarray(Xv[s], np.float32); d = np.diff(x, axis=0, prepend=x[:1])
    o, _, _ = seq_feats(x, Yv[s], 1.0, None)
    ta = np.arange(WARM + PRE, L - max(DS) - 2); cand = ta[d[ta, 10] < THR]
    side0 = d[cand + 1, 5] > -0.05                           # True: ныряет t0
    out = np.zeros((L, 2 * len(LB)), np.float32)
    for D_ in DUSE:
        p = calib(clfs[D_].predict_proba(o[D_])[:, 1])
        u = np.zeros((2, L)); u[0, cand] = p * side0; u[1, cand] = p * ~side0
        C = np.concatenate([np.zeros((2, 1)), np.cumsum(u, 1)], 1)
        for bi, (a, b) in enumerate(LB):
            if DFOR(a) != D_: continue
            lo, hi = np.clip(tt - b + 1, 0, L), np.clip(tt - a + 1, 0, L)
            out[:, 2 * bi:2 * bi + 2] = (C[:, hi] - C[:, lo]).T
    feat3[s] = out
print(f'признаки затухания готовы: {time.time() - t0:.0f} с')

r3 = np.where(famv[qs] == 3)[0]; H = np.zeros((len(r3), 2 * len(LB)), np.float32)
for s in f3_te:
    m = qs[r3] == s; H[m] = feat3[s][qt[r3][m]]
seq3 = qs[r3]
def f3score(P): return np.mean([wcorr(yq[r3, j], P[:, j], np.abs(yq[r3, j])) for j in (0, 1)])

def cv_pred(with_h, lam=1e-3):
    P = np.zeros((len(r3), 2))
    for tr_s, te_s in KFold(5, shuffle=True, random_state=0).split(f3_te):
        tr, te = np.isin(seq3, f3_te[tr_s]), np.isin(seq3, f3_te[te_s])
        Z = np.column_stack([np.ones(len(r3)), base_r[r3]])
        if with_h:
            mu, sd = H[tr].mean(0), H[tr].std(0) + 1e-9; Z = np.column_stack([Z, (H - mu) / sd])
        for j in (0, 1):
            P[te, j] = Z[te] @ wls(Z[tr], yq[r3][tr, j], np.abs(yq[r3][tr, j]), lam)
    return P

Pc, Ph = cv_pred(False), cv_pred(True)
print(f'\nF3 (is_scored): ансамбль {f3score(base_r[r3]):.4f} | контроль {f3score(Pc):.4f} | гибрид {f3score(Ph):.4f}')
full_c, full_h = base_r.copy(), base_r.copy(); full_c[r3], full_h[r3] = Pc, Ph
print(f'все семейства:  ансамбль {sc_all(base_r):.4f} | контроль {sc_all(full_c):.4f} | гибрид {sc_all(full_h):.4f}')
print('гибрид против контроля:', boot(full_c, full_h))
print('ориентир: оракул событий в F3 ≈ 0.52 / 0.56, с задержкой 20 ≈ 0.45 / 0.50')


детекторы обучены: 238 с
признаки затухания готовы: 267 с

F3 (is_scored): ансамбль 0.4022 | контроль 0.3873 | гибрид 0.3791
все семейства:  ансамбль 0.6610 | контроль 0.6518 | гибрид 0.6498
гибрид против контроля: Δ = -0.0021, 90% [-0.0047, +0.0005], P(Δ>0) = 0.09
ориентир: оракул событий в F3 ≈ 0.52 / 0.56, с задержкой 20 ≈ 0.45 / 0.50


In [9]:
%run -i /content/drive/MyDrive/wnn/lib.py
import pyarrow.parquet as pq, subprocess, os, shutil
def ok(p, n):
    try: return pq.ParquetFile(p).num_row_groups == n
    except Exception: return False
NEED = {'valid': 1873, 'train': 10607}
os.makedirs(ROOT, exist_ok=True)
bad = []
for n, rg in NEED.items():
    p = f'{ROOT}/{n}.parquet'
    if os.path.exists(p) and not ok(p, rg): print(f'{n}.parquet битый ({os.path.getsize(p) / 1e9:.1f} ГБ) — удаляю'); os.remove(p)
    if not os.path.exists(p):
        found = subprocess.run(f"find /content/drive/MyDrive -maxdepth 5 -name '{n}.parquet' 2>/dev/null", shell=True,
                               capture_output=True, text=True).stdout.split()
        good = [f for f in found if ok(f, rg)]
        if good: print(f'{n}.parquet: копирую целую копию с Drive ({good[0]})'); shutil.copy(good[0], p)
        else: bad.append(n)
# train нужен только для пересборки train-подвыборок; если их файлы целы, его не качаем
need_train = not all(os.path.exists(f'/content/data/{f}') for f in ('train3k_X.npy', 'train3k_Y.npy'))
to_get = [n for n in bad if n == 'valid' or need_train]
if to_get:
    pats = ' '.join(f"'*/datasets/{n}.parquet'" for n in to_get)
    print('качаю из архива:', to_get, '(около 11 минут, архив читается целиком)')
    subprocess.run(f"curl -sL https://files.wundernn.io/wnn_connectome_starterpack.tar.gz | tar -xz --wildcards {pats} '*/baseline/*' 2>/dev/null",
                   shell=True, cwd='/content')
for n in to_get: assert ok(f'{ROOT}/{n}.parquet', NEED[n]), f'{n}.parquet всё ещё битый'
print('valid.parquet в порядке:', ok(f'{ROOT}/valid.parquet', 1873))

os.makedirs('/content/data', exist_ok=True)
if not all(os.path.exists(f'/content/data/valid300_{s}.npy') for s in 'XYM'):
    build('valid', 'valid300', 300, 1, True)                     # тот же сид, что и раньше
print('файлы в /content/data:', sorted(os.listdir('/content/data')))
print(subprocess.run('df -h /content | tail -1', shell=True, capture_output=True, text=True).stdout)

valid.parquet битый (1.3 ГБ) — удаляю
качаю из архива: ['valid'] (около 11 минут, архив читается целиком)
valid.parquet в порядке: True
valid300 0 0 с
файлы в /content/data: ['train3k_X.npy', 'train3k_Y.npy', 'train_X.npy', 'train_Y.npy', 'valid300_M.npy', 'valid300_X.npy', 'valid300_Y.npy']
overlay         113G   67G   46G  60% /



In [10]:
import torch
Xv = np.load('/content/data/valid300_X.npy', mmap_mode='r'); Yv = np.load('/content/data/valid300_Y.npy')
Mv = np.load('/content/data/valid300_M.npy').astype(bool)
famv = fam_of(Yv, verbose=False); hv = np.bincount(famv, minlength=4) / len(famv)
print('семейства valid300:', np.bincount(famv, minlength=4))

devc = 'cuda' if torch.cuda.is_available() else 'cpu'
net = Net(128).to(devc); net.load_state_dict(torch.load(f'{CKPT_DIR}/gru_full_val_best.pt', map_location=devc)); net.eval()
Pg_saved = np.load(f'{CKPT_DIR}/P_gru6573_valid300.npy', mmap_mode='r')
with torch.no_grad():
    x = torch.from_numpy(np.ascontiguousarray(Xv[:2, :3000], np.float32)).to(devc); o, _ = net(x)
err = np.abs(o.clamp(-2, 2).cpu().numpy() - Pg_saved[:2, :3000]).max()
print(f'сверка с сохранёнными предсказаниями: {err:.1e}'); assert err < 1e-3, 'valid300 собран не так, как раньше'

S = Mv & (np.arange(L) >= WARM)[None]; qs, qt = np.where(S)
yq = np.clip(Yv[qs, qt].astype(np.float64), -2, 2); wq = (VALMIX / hv)[famv[qs]]
PE = np.mean([np.clip(np.load(f'{CKPT_DIR}/Pb_valid300.npy') * 0.3, -2, 2), np.load(f'{CKPT_DIR}/P_gru6573_valid300.npy'),
              np.load(f'{CKPT_DIR}/P_corr3k_valid300.npy')], 0).astype(np.float64)
base_r = PE[qs, qt]
def wcorr(y, p, w):
    sw = w.sum(); my, mp = (w * y).sum() / sw, (w * p).sum() / sw
    return (w * (y - my) * (p - mp)).sum() / np.sqrt((w * (y - my) ** 2).sum() * (w * (p - mp) ** 2).sum())
print('ансамбль на is_scored:', round(np.mean([wcorr(yq[:, j], base_r[:, j], np.abs(yq[:, j]) * wq) for j in (0, 1)]), 4), '(ожидаю 0.6610)')

семейства valid300: [68 54 80 98]
сверка с сохранёнными предсказаниями: 0.0e+00
ансамбль на is_scored: 0.661 (ожидаю 0.6610)


In [11]:
import numpy as np, torch, time
Xv = np.load('/content/data/valid300_X.npy', mmap_mode='r'); Yv = np.load('/content/data/valid300_Y.npy')
devc = 'cuda' if torch.cuda.is_available() else 'cpu'
BLK = {'i0 цены': range(0, 22), 'i0 объёмы': range(22, 44), 'i0 сделки': range(44, 52), 'i1 цены': range(52, 74),
       'i1 объёмы': range(74, 96), 'i1 сделки': range(96, 104), 'a0–a7': range(104, 112)}

# ---------- 1. распределения входов и float16 ----------
S = np.concatenate([np.asarray(Xv[s, ::10], np.float32) for s in range(0, 300, 10)])        # 30 посл., каждый 10-й шаг
q = np.quantile(S, [0.001, 0.01, 0.25, 0.5, 0.75, 0.99, 0.999], axis=0); sd = S.std(0); iqr = q[4] - q[2]
mx = np.abs(np.asarray(Xv[:, ::50], np.float32)).max((0, 1))
print('макс |x| по всем 300 посл.:', float(mx.max()), '| переполнение float16:', bool((mx > 65504).any()))
STUB = -5.199337
print('\nблок        | медиана IQR/std | колонок с IQR/std<0.1 | медиана q99.9/q99 | колонок с массой на −5.2 / +5.2')
for b, cols in BLK.items():
    c = list(cols); r = iqr[c] / (sd[c] + 1e-9); tail = (q[6][c] - q[3][c]) / (q[5][c] - q[3][c] + 1e-9)
    lo = sum((np.isclose(S[:, k], STUB, atol=1e-4).mean() > 0.01) for k in c); hi = sum((np.isclose(S[:, k], -STUB, atol=1e-4).mean() > 0.01) for k in c)
    print(f'{b:11s} | {np.median(r):15.3f} | {(r < 0.1).sum():>21} | {np.median(tail):17.2f} | {lo} / {hi}')
print('\nстолбцы с самым сжатым телом (IQR/std):', [(int(k), round(float(iqr[k] / sd[k]), 3)) for k in np.argsort(iqr / (sd + 1e-9))[:10]])
nrm = [int(k) for k in range(112) if abs(q[5][k] - 2.326) < 0.05 and abs(q[2][k] + 0.674) < 0.05]
print('колонки, похожие на точную N(0,1) (q25 ≈ −0.674, q99 ≈ 2.326):', nrm)

# ---------- 2. спад по позиции ----------
Pg = np.load(f'{CKPT_DIR}/P_gru6573_valid300.npy'); Pb3 = np.clip(np.load(f'{CKPT_DIR}/Pb_valid300.npy') * 0.3, -2, 2)
BINS = ((99, 2000), (2000, 5000), (5000, 10000), (10000, 15000), (15000, 20000))
sc_bin = lambda P, m: np.mean([wcorr(yq[m, j], P[m, j], np.abs(yq[m, j]) * wq[m]) for j in (0, 1)])
print('\nкорзина        | ансамбль | наша GRU | baseline | std таргета | разброс уровней цен i0 | средний |Δ| цен i0')
for lo, hi in BINS:
    m = (qt >= lo) & (qt < hi)
    lev = np.asarray(Xv[:, lo:hi:20, :22], np.float32).mean(1)                              # средний уровень по посл.
    dif = np.abs(np.diff(np.asarray(Xv[:, lo:hi:5, :22], np.float32), axis=1)).mean()
    ystd = np.sqrt(np.average(yq[m] ** 2, weights=np.abs(yq[m]) * wq[m][:, None], axis=0)).mean()
    print(f'{lo:>5}–{hi:<6} | {sc_bin(base_r, m):8.4f} | {sc_bin(Pg[qs, qt], m):8.4f} | {sc_bin(Pb3[qs, qt], m):8.4f} | '
          f'{ystd:11.3f} | {lev.std(0).mean():22.3f} | {dif:.4f}')

# сброс состояния на шаге 9000: скор корзины 10 000–15 000 с «свежей» GRU против обычной
net = Net(128).to(devc); net.load_state_dict(torch.load(f'{CKPT_DIR}/gru_full_val_best.pt', map_location=devc)); net.eval()
Pr = np.zeros((300, 6000, 2), np.float32); t0 = time.time()
with torch.no_grad():
    for s in range(0, 300, 50):
        x = torch.from_numpy(np.ascontiguousarray(Xv[s:s + 50, 9000:15000], np.float32)).to(devc); h = None; o_ = []
        for t in range(0, 6000, 2000): o, h = net(x[:, t:t + 2000], h); o_.append(o)
        Pr[s:s + 50] = torch.cat(o_, 1).clamp(-2, 2).cpu().numpy()
m = (qt >= 10000) & (qt < 15000)
print(f'\nсброс на 9000 ({time.time() - t0:.0f} с): корзина 10 000–15 000 | обычная GRU {sc_bin(Pg[qs, qt], m):.4f} | '
      f'со сбросом {sc_bin(Pr[qs[m], qt[m] - 9000], np.ones(m.sum(), bool)) if False else np.mean([wcorr(yq[m, j], Pr[qs[m], qt[m] - 9000, j], np.abs(yq[m, j]) * wq[m]) for j in (0, 1)]):.4f}')

# ---------- 3. расшифровка таргета ----------
U = np.array([0.1254, 0.2413, 0.4685, 0.6569])
for f in (3, 2):
    ids = np.where(famv == f)[0]; Y = np.clip(Yv[ids, WARM:], -2, 2)
    ssum = np.round((Y[..., 0] + Y[..., 1]) / U[f]).astype(int).ravel(); v, c = np.unique(ssum, return_counts=True); o = np.argsort(-c)
    print(f'\nF{f}: (t0 + t1)/u — топ значений и доли:', [(int(v[i]), round(c[i] / c.sum(), 3)) for i in o[:8]], f'| уникальных {len(v)}')
    ch = np.zeros(0, bool); still = np.zeros(0, bool)
    for s in ids[:40]:
        y = Yv[s, WARM:]; x = np.asarray(Xv[s, WARM:, :22], np.float32)
        ch = np.r_[ch, (np.diff(y, axis=0) != 0).any(1)]; still = np.r_[still, (np.diff(x, axis=0) == 0).all(1)]
    print(f'F{f}: доля шагов без изменения всех цен i0: {still.mean():.3f} | доля смен таргета на таких шагах: {(ch & still).sum() / max(ch.sum(), 1):.3f}')

макс |x| по всем 300 посл.: 5.199337482452393 | переполнение float16: False

блок        | медиана IQR/std | колонок с IQR/std<0.1 | медиана q99.9/q99 | колонок с массой на −5.2 / +5.2
i0 цены     |           0.548 |                     0 |              1.21 | 0 / 0
i0 объёмы   |           1.358 |                     0 |              1.30 | 0 / 0
i0 сделки   |           0.301 |                     4 |              1.17 | 0 / 0
i1 цены     |           0.719 |                     1 |              1.36 | 0 / 0
i1 объёмы   |           1.071 |                     0 |              1.22 | 0 / 0
i1 сделки   |           0.270 |                     4 |              1.01 | 0 / 0
a0–a7       |           1.285 |                     0 |              1.21 | 3 / 0

столбцы с самым сжатым телом (IQR/std): [(102, 0.016), (55, 0.038), (103, 0.057), (100, 0.058), (101, 0.061), (51, 0.086), (48, 0.088), (50, 0.093), (49, 0.094), (5, 0.232)]
колонки, похожие на точную N(0,1) (q25 ≈ −0.674, q99 ≈ 2.326): []


In [12]:
from sklearn.ensemble import HistGradientBoostingClassifier
U = np.array([0.1254, 0.2413, 0.4685, 0.6569]); R = len(qs); uf = U[famv[qs]]
Sraw = (Yv[..., 0] + Yv[..., 1]).astype(np.float64); S_r = Sraw[qs, qt]
k_r = np.round(S_r / uf).astype(int)
sc = lambda P, m=slice(None): np.mean([wcorr(yq[m, j], P[m, j], np.abs(yq[m, j]) * wq[m]) for j in (0, 1)])

print('S = t0 + t1 в единицах u на is_scored:')
for f in range(4):
    v, c = np.unique(k_r[famv[qs] == f], return_counts=True); o = np.argsort(-c)
    print(f'  F{f}:', [(int(v[i]), round(c[i] / c.sum(), 3)) for i in o[:5]])

p0, p1 = base_r[:, 0], base_r[:, 1]; Dy = yq[:, 0] - yq[:, 1]; wS = wq * (np.abs(yq[:, 0]) + np.abs(yq[:, 1])) / 2
print(f'\nансамбль угадывает сумму: corr(p0+p1, S) = {wcorr(S_r, p0 + p1, wS):.3f} | разность: corr(p0−p1, D) = {wcorr(Dy, p0 - p1, wS):.3f}')
print(f'ансамбль как есть: {sc(base_r):.4f}')

# ---------- потолок: истинное S + ансамблевая разность ----------
print('\nпотолок «истинное S + β·(p0 − p1)»:')
for b in (0.5, 0.75, 1.0, 1.25, 1.5, 2.0):
    P = np.column_stack([(S_r + b * (p0 - p1)) / 2, (S_r - b * (p0 - p1)) / 2]); print(f'  β={b:<5} {sc(P):.4f}')

# ---------- восстанавливается ли S по текущему состоянию ----------
Xr = np.zeros((R, 225), np.float32); t0_ = time.time()
for s in range(300):
    r = np.where(qs == s)[0]
    if not len(r): continue
    x = np.asarray(Xv[s], np.float32); d = np.diff(x, axis=0, prepend=x[:1])
    Xr[r, :112], Xr[r, 112:224] = x[qt[r]], d[qt[r]]; Xr[r, 224] = famv[s]
print(f'\nпризнаки строк: {time.time() - t0_:.0f} с')
half = (np.arange(300) % 2 == 0); Sh = np.zeros(R); acc = []
rng = np.random.default_rng(0)
for a in (True, False):
    tr, te = np.where(half[qs] == a)[0], np.where(half[qs] != a)[0]
    sub = rng.choice(tr, min(300_000, len(tr)), replace=False)
    clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, max_leaf_nodes=63, random_state=0).fit(Xr[sub], k_r[sub])
    pr = clf.predict_proba(Xr[te]); Sh[te] = (pr @ clf.classes_.astype(np.float64)) * uf[te]
    acc.append((clf.classes_[pr.argmax(1)] == k_r[te]).mean())
print('точность класса S по текущему состоянию (две половины):', np.round(acc, 3))
for f in range(4):
    m = famv[qs] == f; print(f'  F{f}: corr(Ŝ, S) = {np.corrcoef(Sh[m], S_r[m])[0, 1]:.3f} | у ансамбля corr(p0+p1, S) = {np.corrcoef(p0[m] + p1[m], S_r[m])[0, 1]:.3f}')

print('\nчестный вариант «Ŝ по признакам + β·(p0 − p1)»:')
for b in (0.75, 1.0, 1.25, 1.5):
    P = np.column_stack([(Sh + b * (p0 - p1)) / 2, (Sh - b * (p0 - p1)) / 2]); print(f'  β={b:<5} {sc(P):.4f}')

S = t0 + t1 в единицах u на is_scored:
  F0: [(-3, np.float64(0.147)), (-2, np.float64(0.138)), (-4, np.float64(0.12)), (-1, np.float64(0.094)), (-5, np.float64(0.071))]
  F1: [(-2, np.float64(0.28)), (-1, np.float64(0.241)), (-3, np.float64(0.114)), (0, np.float64(0.076)), (1, np.float64(0.059))]
  F2: [(-1, np.float64(0.565)), (-2, np.float64(0.299)), (0, np.float64(0.048)), (1, np.float64(0.03)), (-3, np.float64(0.029))]
  F3: [(-1, np.float64(0.748)), (-2, np.float64(0.189)), (0, np.float64(0.037)), (1, np.float64(0.014)), (2, np.float64(0.006))]

ансамбль угадывает сумму: corr(p0+p1, S) = 0.497 | разность: corr(p0−p1, D) = 0.618
ансамбль как есть: 0.6610

потолок «истинное S + β·(p0 − p1)»:
  β=0.5   0.6687
  β=0.75  0.6920
  β=1.0   0.6976
  β=1.25  0.6960
  β=1.5   0.6917
  β=2.0   0.6814

признаки строк: 2 с


ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

In [13]:
from sklearn.ensemble import HistGradientBoostingRegressor
kc = np.clip(k_r, -8, 4)                                        # редкие крайние значения в крайний класс
print('доля строк, затронутых обрезкой классов:', round(float((kc != k_r).mean()), 4))
half = (np.arange(300) % 2 == 0); Sh = np.zeros(R); Sg = np.zeros(R); acc = []
rng = np.random.default_rng(0)
for a in (True, False):
    tr, te = np.where(half[qs] == a)[0], np.where(half[qs] != a)[0]
    sub = rng.choice(tr, min(300_000, len(tr)), replace=False); t0_ = time.time()
    clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, max_leaf_nodes=63, early_stopping=False,
                                         random_state=0).fit(Xr[sub], kc[sub])
    pr = clf.predict_proba(Xr[te]); Sh[te] = (pr @ clf.classes_.astype(np.float64)) * uf[te]
    acc.append((clf.classes_[pr.argmax(1)] == kc[te]).mean())
    reg = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1, max_leaf_nodes=63, early_stopping=False,
                                        random_state=0).fit(Xr[sub], S_r[sub])
    Sg[te] = reg.predict(Xr[te]); print(f'  половина {int(a)}: {time.time() - t0_:.0f} с', flush=True)

print('точность класса S по текущему состоянию (две половины):', np.round(acc, 3))
for f in range(4):
    m = famv[qs] == f
    print(f'  F{f}: corr(Ŝ класс., S) = {np.corrcoef(Sh[m], S_r[m])[0, 1]:.3f} | corr(Ŝ регр., S) = {np.corrcoef(Sg[m], S_r[m])[0, 1]:.3f} '
          f'| у ансамбля corr(p0+p1, S) = {np.corrcoef(p0[m] + p1[m], S_r[m])[0, 1]:.3f}')

print('\nчестный вариант «Ŝ по признакам + β·(p0 − p1)»:')
for name, Sx in (('классификатор', Sh), ('регрессия', Sg)):
    for b in (0.75, 1.0, 1.25):
        P = np.column_stack([(Sx + b * (p0 - p1)) / 2, (Sx - b * (p0 - p1)) / 2]); print(f'  {name:13s} β={b:<5} {sc(P):.4f}')
for w in (0.25, 0.5, 0.75):                                     # смесь: сумма частично из ансамбля, частично из Ŝ
    Sx = w * Sh + (1 - w) * (p0 + p1)
    P = np.column_stack([(Sx + (p0 - p1)) / 2, (Sx - (p0 - p1)) / 2]); print(f'  смесь сумм, доля Ŝ = {w:<5} {sc(P):.4f}')
print(f'ориентиры: ансамбль {sc(base_r):.4f} | потолок с истинным S 0.6976')

доля строк, затронутых обрезкой классов: 0.0741
  половина 1: 1489 с


KeyboardInterrupt: 

In [14]:
m1 = ~half[qs]                                                  # тест первой половины: нечётные последовательности
print('строк в оценке:', int(m1.sum()), '| точность класса S:', round(float(acc[0]), 3) if acc else '—')
for f in range(4):
    m = m1 & (famv[qs] == f)
    print(f'  F{f}: corr(Ŝ класс., S) = {np.corrcoef(Sh[m], S_r[m])[0, 1]:.3f} | corr(Ŝ регр., S) = {np.corrcoef(Sg[m], S_r[m])[0, 1]:.3f} '
          f'| у ансамбля corr(p0+p1, S) = {np.corrcoef(p0[m] + p1[m], S_r[m])[0, 1]:.3f}')

print(f'\nна этих строках: ансамбль {sc(base_r, m1):.4f}')
Pt = np.column_stack([(S_r + (p0 - p1)) / 2, (S_r - (p0 - p1)) / 2]); print(f'потолок с истинным S: {sc(Pt, m1):.4f}')
for name, Sx in (('классификатор', Sh), ('регрессия', Sg)):
    for b in (0.75, 1.0, 1.25):
        P = np.column_stack([(Sx + b * (p0 - p1)) / 2, (Sx - b * (p0 - p1)) / 2]); print(f'  {name:13s} β={b:<5} {sc(P, m1):.4f}')
for w in (0.25, 0.5, 0.75):
    Sx = w * Sg + (1 - w) * (p0 + p1)
    P = np.column_stack([(Sx + (p0 - p1)) / 2, (Sx - (p0 - p1)) / 2]); print(f'  смесь сумм (регр.), доля Ŝ = {w:<5} {sc(P, m1):.4f}')

строк в оценке: 292286 | точность класса S: 0.524
  F0: corr(Ŝ класс., S) = 0.233 | corr(Ŝ регр., S) = 0.292 | у ансамбля corr(p0+p1, S) = 0.250
  F1: corr(Ŝ класс., S) = 0.105 | corr(Ŝ регр., S) = 0.267 | у ансамбля corr(p0+p1, S) = 0.328
  F2: corr(Ŝ класс., S) = 0.157 | corr(Ŝ регр., S) = 0.376 | у ансамбля corr(p0+p1, S) = 0.291
  F3: corr(Ŝ класс., S) = 0.053 | corr(Ŝ регр., S) = 0.138 | у ансамбля corr(p0+p1, S) = 0.027

на этих строках: ансамбль 0.6645
потолок с истинным S: 0.6999
  классификатор β=0.75  0.6132
  классификатор β=1.0   0.6217
  классификатор β=1.25  0.6242
  регрессия     β=0.75  0.6511
  регрессия     β=1.0   0.6484
  регрессия     β=1.25  0.6445
  смесь сумм (регр.), доля Ŝ = 0.25  0.6661
  смесь сумм (регр.), доля Ŝ = 0.5   0.6643
  смесь сумм (регр.), доля Ŝ = 0.75  0.6585


In [15]:
PS = np.zeros_like(PE); stats = []
for s in range(300):
    x = np.asarray(Xv[s, :, :22], np.float32)
    ch = np.r_[True, (np.diff(x, axis=0) != 0).any(1)]           # шаг, на котором сменилась хоть одна цена i0
    seg = np.cumsum(ch) - 1; st = np.flatnonzero(ch)[seg]          # начало текущего сегмента для каждого шага
    cs = np.cumsum(PE[s], 0); prev = np.where(st[:, None] > 0, cs[np.maximum(st - 1, 0)], 0.0)
    PS[s] = (cs - prev) / (np.arange(L) - st + 1)[:, None]         # причинное среднее прогноза внутри сегмента
    yc = np.r_[False, (np.diff(Yv[s], axis=0) != 0).any(1)]
    stats.append((famv[s], ch[WARM:].mean(), (yc & ch)[WARM:].sum() / max(ch[WARM:].sum(), 1)))
st_ = np.array(stats)
for f in range(4):
    g = st_[st_[:, 0] == f]
    print(f'F{f}: цены i0 меняются на {g[:, 1].mean():.3f} шагов | на ценовых шагах таргет меняется в {g[:, 2].mean():.3f} случаев')

print(f'\nансамбль как есть: {sc(PE[qs, qt]):.4f}')
for lam in (0.25, 0.5, 0.75, 1.0):
    P = (1 - lam) * PE + lam * PS; print(f'  доля среднего по сегменту λ={lam:<5} {sc(P[qs, qt]):.4f}')

F0: цены i0 меняются на 0.440 шагов | на ценовых шагах таргет меняется в 0.529 случаев
F1: цены i0 меняются на 0.265 шагов | на ценовых шагах таргет меняется в 0.549 случаев
F2: цены i0 меняются на 0.108 шагов | на ценовых шагах таргет меняется в 0.560 случаев
F3: цены i0 меняются на 0.327 шагов | на ценовых шагах таргет меняется в 0.649 случаев

ансамбль как есть: 0.6610
  доля среднего по сегменту λ=0.25  0.6585
  доля среднего по сегменту λ=0.5   0.6516
  доля среднего по сегменту λ=0.75  0.6396
  доля среднего по сегменту λ=1.0   0.6219


In [16]:
%run -i /content/drive/MyDrive/wnn/lib.py
assert torch.cuda.is_available(), 'нужен GPU'
Xtr = np.load('/content/data/train3k_X.npy', mmap_mode='r'); Ytr = np.load('/content/data/train3k_Y.npy')
Xva = np.load('/content/data/valid300_X.npy', mmap_mode='r'); Yva = np.load('/content/data/valid300_Y.npy'); Mva = np.load('/content/data/valid300_M.npy')
fam_v = fam_of(Yva, verbose=False); have = np.bincount(fam_v, minlength=4) / len(fam_v)
print('обучающих:', len(Xtr))
torch.manual_seed(0)
net15 = train(Net().to(dev), epochs=15, tag='gru3k_e15')
torch.save(net15.state_dict(), f'{CKPT_DIR}/gru3k_e15_last.pt')
print('15 эпох, последняя:', mix_score(net15), '| ориентир: 3000 посл., 8 эпох — 0.488 / 0.654')

обучающих: 3000


/content/drive/MyDrive/wnn/lib.py:75: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  x = torch.from_numpy(np.ascontiguousarray(X[s:s + bs])).to(dev)


эпоха 1/15 | 164 с | loss 1.0621 | valid все строки 0.448 [0.446 0.449] | is_scored 0.644 [0.64  0.647]
эпоха 2/15 | 168 с | loss 1.0037 | valid все строки 0.471 [0.467 0.474] | is_scored 0.649 [0.646 0.652]
эпоха 3/15 | 170 с | loss 0.9933 | valid все строки 0.470 [0.47  0.471] | is_scored 0.646 [0.641 0.65 ]
эпоха 4/15 | 167 с | loss 0.9882 | valid все строки 0.476 [0.472 0.479] | is_scored 0.660 [0.653 0.666]
эпоха 5/15 | 169 с | loss 0.9845 | valid все строки 0.476 [0.472 0.48 ] | is_scored 0.656 [0.654 0.658]
эпоха 6/15 | 169 с | loss 0.9761 | valid все строки 0.479 [0.482 0.475] | is_scored 0.657 [0.651 0.664]
эпоха 7/15 | 168 с | loss 0.9670 | valid все строки 0.472 [0.476 0.468] | is_scored 0.660 [0.656 0.665]
эпоха 8/15 | 169 с | loss 0.9608 | valid все строки 0.486 [0.486 0.487] | is_scored 0.664 [0.659 0.669]
эпоха 9/15 | 169 с | loss 0.9458 | valid все строки 0.480 [0.472 0.487] | is_scored 0.656 [0.65  0.663]
эпоха 10/15 | 171 с | loss 0.9304 | valid все строки 0.483 [0.48